# `C10` — Fill-or-kill, in three parts

Scenario **C10** of the Track A certification ladder (card `OMIE-CERT-C10-1`, [`agile-plan.md`](../../docs/omiecid/agile-plan.md); canonical spec [`scenario-ladder.md`](../../docs/omiecid/scenario-ladder.md) §C10). Oracle kind: **exact-parametric, plus M4**. **High-risk scenario** (independence rule 4: C10c, the pre-check projection).

**Terms used below** (defined once in `scenario-ladder.md`, "What eager, jitted and assertion mean here"): *eager* runs every JAX operation immediately (`jax.disable_jit()`), so a human can inspect each step; *jitted* traces and compiles the whole run (`jax.jit`), the mode training uses; the two must agree. An *assertion* records a failure and the notebook carries on: every check is evaluated, and the verdict comes at the end. Every pass/fail assertion here is an **exact integer equality** (or membership in an explicitly listed set, or structural equality of two books) on order ids, prices in ticks (0.01 EUR/MWh), quantities in ticks (0.1 MW) and submission indices. Euro figures are shown, never asserted.

## 1. Purpose

**Purpose.** Fill-or-kill has the most dangerous failure mode in the engine, because it requires the
simulator to *look before it leaps*: count available volume first, then decide whether to match at
all. Two things can go wrong. The count can be wrong — and OMIE's rules make it genuinely subtle,
because the pre-check may see volume **hidden from the public book**, and when the iceberg carries a
price increment the pre-check must project the price ladder forward slice by slice to know how much
is actually reachable. Or the rollback can be wrong: a partial match applied and imperfectly undone,
leaving the book corrupted with no trade to show for it. The three parts separate these concerns, and
M4 pins the rollback by requiring the rejected case to leave the book byte-identical.
`tests/omiecid/test_fok_icewpi_precheck.py` is the locked predecessor this generalises.

This scenario deliberately **follows** the iceberg scenarios. Parts (b) and (c) depend on
hidden-volume accounting and the drifted price ladder that C8 and C9 certify; C10c's oracle is
literally C9's price sequence summed, so running it first would make a failure unattributable between
pre-check counting and drift computation — the very separation this scenario exists to provide.
Parts (b) and (c) keep every iceberg within the slice margin of the capacity-validity
precondition, $\lceil \mathit{total}/\mathit{peak} \rceil \le 64$. Past it the registration raises `truncated`, so the draw
would be `INVALID` before the pre-check is ever exercised.

*(Verbatim from `scenario-ladder.md` §C10.)*

**Certifies** (card): §4.5 FOK row; §4.2.1 Rule 1 (the load-bearing one) and Rule 2 (multi-slice per aggressive order); §6.2 item 7 (no price capping); OQ-11. **Pass** (card): exact accept/reject and fill quantity; on reject, structural equality of the pre- and post-submit book; the eager and jitted runs agree; the full battery runs; I8 capacity headroom; I9 non-vacuity; every quote matches its source byte for byte. **Failure implicates:** pre-check counting (a/b), price projection (c), or rollback (M4).

**The four parts** (C10a–C10c are constructions of the generator; C10d is built in section 4 of this notebook from the rules, finding F14; each draw on a fresh book of one contract; the FOK is always the last submission):

| part | construction | the resting side | the FOK | known result (card) |
|---|---|---|---|---|
| **C10a** boundary | `C10.fok_visible` | a visible NON ladder, $L \in [1, 5]$ price levels, depth $V$ | the other side, limit at the deepest level, request $V + \delta$ | the M4 dichotomy, both sides every draw |
| **C10b** hidden volume | `C10.fok_hidden` | one standard iceberg ($\mathit{ppd} = 0$), `(peak, total)`, $\lceil \mathit{total}/\mathit{peak} \rceil \le 64$ | priced through the iceberg, request $\mathit{req}$ | executed iff $\mathit{req} \le \mathit{total}$: hidden quantity counts |
| **C10c** price-incremented | `C10.fok_icewpi` | one ICEwPI iceberg `(base, ppd, peak, total)`, sign-valid, $\lceil \mathit{total}/\mathit{peak} \rceil \le 64$ | limit on the drifted ladder | matchable $= \min(\mathit{total},\ \mathit{peak}\times\#\{k \ge 0 : \mathit{base}+k\,\mathit{ppd} \text{ crosses the limit}\})$; on execution, fills at the drifted prices, uncapped |
| **C10d** iceberg + worse level | `C10.fok_ice_worse_level` (notebook-local, finding F14) | one iceberg (standard, or sign-valid ICEwPI with $r \in [2, n-1]$ crossing slices) **and behind it** one visible NON level at a strictly worse price, $q_v$ | limit at or through the visible level, request around $T_c$ and $\mathcal{M}$ | matchable $\mathcal{M} = T_c + q_v$, $T_c = \min(\mathit{total},\ \mathit{peak}\cdot r)$; on execution, fills best price first, the iceberg's crossing slices and the visible level merged by rank (a slice tying the visible price comes after it), each at its maker's price |

**Both sides of the boundary, every draw** (finding F3). Each draw is run at more than one request: the drawn tape, plus **twin tapes** that differ from it **only** in the FOK row's quantity, so every draw shows the FOK executed at the boundary and killed one tick past it. C10a runs $\{V-1,\ V,\ V+1\}$; C10b runs $\{\mathit{req},\ \mathit{total},\ \mathit{total}+1\}$; C10c runs $\{M,\ M+1\}$, $M$ the projected matchable volume; C10d runs $\{T_c,\ T_c+1,\ \mathcal{M},\ \mathcal{M}+1\}$ (the iceberg's crossing volume exactly, one tick into the worse level, everything, one tick past it). The drawn request is always one of them.

Per run, in one combined assertion per draw whose expected and observed values are recorded **in full** (section 7): **(V)** the verdict, *executed* (filled exactly the request) or *killed* (no fill at all); **(A)** every order accepted, none rejected — a killed FOK is an accepted order cancelled whole (§5.1), finding F2; **(Q)** every trade row `[buyer, seller, price, qty, submission]` in row order (D3-08: the counterparty sequence is pinned), each at its maker's price — the level's in C10a, the slice's (drifted) price in C10b/C10c, the slice's or the visible level's in C10d, never the FOK limit; **(F)** filled and remaining quantity per order id; the FOK never rests; **(C)** the cancelled quantity per order id and cause (the killed FOK's whole quantity as `fok_kill`; nothing else), no eviction; **(B)** the final book, one of the listed books (finding F6); **(R)** on a kill, the book **before** the FOK's submission and the book **after** it are structurally equal (finding F5). Then the relation **M4** from the battery on every run, the battery (RB, I1–I9) on every run, explicit I3 and I5 assertions (a check fails only on `status == "fail"`; `not_applicable` is vacuously satisfied and its counts are reported, finding F13), the capacity checks, the discrimination checks (finding F9), coverage and the eager comparisons.

**K2 findings** (recorded here; none blocks the expectation):

- **F1 — spec version.** The card names `market-spec.md` v2.12; the file is now **v2.14** (v2.13 touched §4.2.1 Rule 3 and §6.2 item 7 only, OQ-31; v2.14 is additive: no quoted byte changed). v2.14 adds two passages that bear on C10: the Rule 2 *Re-ranking* clarification (quoted in section 2; it decides C10d's fill order) and the Rule 4 *Slice activation timing* correction note (activation is eager in the market; whether a lazily activated slice makes an observable difference is left to the architect). F6's one-of-two final book is kept as written, because the correction note explicitly does not reopen the OQ-20 engineering conclusion; recorded here so a reviewer can tighten it. Every quote is checked against the current file at run time.
- **F2 — "accept / reject" means executed / killed.** §5.1 puts an unmatched FOK under `ACCEPTED → CANCELLED (IOC/FOK unmatched; …)`, not under `REJECTED`. So a killed FOK is expected among the **accepted** orders, with its whole quantity cancelled as `fok_kill`, zero fills and no book entry (§4.5: "No (cancelled)"); `rejected` is expected empty on every run. This is also what the battery's M4 and I2 assume. An engine that reports a killed FOK as a validation rejection fails (A) and (C); that would be a reading to escalate, not to absorb.
- **F3 — "both sides every draw".** The generator draws one request per seed ($\delta \in \{-1, 0, +1\}$ in C10a; $\mathit{req} \in [\mathit{peak}+1, \mathit{total}]$ or $\mathit{total}+1$ in C10b; $M$ or $M+1$ in C10c). The card asks for the dichotomy **both sides, every draw**, so the twins above are built in section 4 from the tape alone (one integer changes: the FOK row's quantity; the capacity bounds are recomputed from the twin tape and re-checked). They are counterfactual inputs of this notebook, not generator modes; no generator code is changed.
- **F4 — the C10c formula, both sides.** The card's $\#\{k \ge 0 : \mathit{base}+k\,\mathit{ppd} \le \mathit{limit}\}$ is the SELL-iceberg form (a BUY FOK crosses a sell slice iff its price $\le$ the limit, §6.2 item 2). For a BUY iceberg (a SELL FOK) the test is $\ge$. With a sign-valid increment the slices drift away from the FOK, so the crossing slices form a prefix and the count is finite; $\min(\mathit{total}, \cdot)$ caps it at the iceberg (and would absorb a short last slice). Section 5 computes the count both ways — by the closed form from `draw.params` and by projecting the tape's ladder — and asserts they agree on every draw.
- **F5 — what "structural equality of the pre- and post-submit book" compares.** The record builder's snapshot just before the FOK's submission and just after it: the bid and the ask arrays byte for byte (every column, the stored timestamps included), the trade ring byte for byte, the hidden quantity of every iceberg in the draw, and the `book_full` / `truncated` flags. The battery's M4 is asserted **as well**, with its own definition (it also compares the iceberg side-channel array and its live count). Whether a killed FOK consumes a number of the engine's timestamp counter is not ruled by the spec (§5.2: a timestamp is received "when entering the SOB", which a FOK never does); the counter is **reported, not asserted**.
- **F6 — the final book after an executed FOK may be one of two.** When the FOK ends exactly on a slice boundary of an iceberg that still has hidden quantity, the next slice is either shown (at its own price: the base for C10b, $\mathit{base} + k\,\mathit{ppd}$ for C10c, with $\min(\mathit{peak}, \text{what is left})$) or not yet shown — the permitted between-aggressor state (§4.2.1 Scope clarification; OQ-20). The final book is asserted to be **one of** the two; the iceberg's remaining quantity (visible + hidden) is exact in both. Which one occurred is reported, never asserted. A killed FOK leaves the book as it was, so there is exactly one book.
- **F7 — passive/aggressor labels are not asserted.** As in C08 and C09, the rules fix the price-setter, not the engine's passive label inside a multi-slice match (OQ-31's open engine-facing item (b)); trades are pinned by buyer and seller order ids, price, quantity and submission.
- **F8 — where the price-setter claim discriminates.** C10a's FOK limit is the deepest level's price, so it discriminates maker pricing on levels $0..L-2$ only (draws with $L \ge 2$). C10b's FOK is 1–100 ticks through the iceberg, so every trade discriminates. C10c's limit is exactly the last crossing slice's price, so slices $0..r-2$ discriminate (draws with $r \ge 2$). The classes with $L \ge 2$ and $r \ge 2$ are required by the coverage check.
- **F9 — every draw catches the wrong readings (non-vacuity of the oracle, rule 1).** Per draw, from the tape alone, section 5 evaluates named wrong pre-checks and asserts each gives a different verdict than the rule on at least one of the draw's runs: C10a — a count one tick short and one tick long; C10b — a count of the visible peak only, and one tick long; C10c — OQ-11's **Interpretation A** (all remaining quantity if the base price crosses), one slice too many, and one slice too few ($(r-1)\,\mathit{peak}$, which kills the request $M$). A capped fill (Interpretation A's second half) is caught by the request $M + 1$, which Interpretation A executes.
- **F10 — a capacity flag inside the margin is a failure, not only a discard** (as C08 F7, C09 F9). A run whose tape passed the pre-run margin (hop bound ≤ 160, ≤ 80 trades, ≤ 80 resting per side, ≤ 800 icebergs, every $\lceil \mathit{qty}/\mathit{peak} \rceil \le 64$) and still raised a capacity flag discards its draw **and** is recorded as a failed assertion.
- **F11 — the draw envelope is the generator's, empirical.** C10a: 1–5 levels, 1.0–15.0 MW each, 1–20 ticks apart, base 30–90 EUR/MWh. C10b: peak 1.0–5.0 MW, 2–14 slices, FOK 1–100 ticks through. C10c: $|\mathit{ppd}|$ 1–29 ticks (0.01–0.29 EUR/MWh), 3–14 full slices, $r \in [1, n-1]$ crossing slices. These are test ranges, never bounds (§13.6 for the increment). The ranges actually drawn are reported in section 4. **Not drawn:** a C10a FOK whose limit excludes some resting levels (visible price filtering), and requests at or below one visible peak in C10b; recorded as coverage limits, not claims.
- **F12 — what the engine reports is not asserted here.** Report-equals-tape is the battery's I6(b); it runs on every run. `draw.expected` is never read.
- **F13 — `not_applicable` is vacuously satisfied, never a failure** (OMIE-CERT-RERUN-1; certification-log entry 11). A killed FOK writes no trade, so the battery's I5 (and I1, I4) return `not_applicable` on it: §6.2 items 3 and 7 say nothing about a run without trades. The explicit I3, I5 and M4 aggregates of section 7 therefore fail **only** on `status == "fail"` (never on `!= "pass"`), and report the `not_applicable` count per check, part and expected verdict, with every `part seed:request` listed in full. A `not_applicable` I5 on an executed run cannot hide a defect: that run has no trade, which the per-draw (V)/(Q) assertion already fails.
- **F14 — C10d, the FOK against an iceberg plus a worse visible level** (the correction's new construction). Three rows on side $\sigma$ then $\bar\sigma$: an iceberg (submission 0), one visible NON at a price **strictly worse** than the iceberg's base (submission 1), the FOK (submission 2) with its limit at or through that level. Two ladder kinds: a standard iceberg ($\mathit{ppd} = 0$, a short last slice allowed), so every slice is better than the visible level; and a sign-valid ICEwPI with $r \in [2, n-1]$ crossing slices, whose drifted slices can fall **behind** the visible level (the visible level then trades between slices). The pre-check must add the visible level to the iceberg's crossing slices; the fill must take them in rank order. It is built from the rules in section 4 (seeded, `numpy.random.default_rng`), not by the generator: no generator code is changed. It therefore has no registry construction digest (the results JSON lists only C10a–C10c's); its provenance is the tape kept by `record_case` and the section-4 code.
- **F15 — C10d's tie.** When a drifted slice $k \ge 1$ has exactly the visible level's price, the visible order trades first: the slice is a new entry with a new timestamp, activated during the FOK's own matching process, after the visible order entered (§4.2: "receives a new timestamp"; §4.2.1 Rule 2 *Re-ranking*: "when the slice price equals it and the other order has the earlier timestamp"; §6.1). Slice 0 is older than the visible order (it was submitted first), but it is always strictly better priced.
- **F16 — what C10d discriminates.** Named wrong pre-checks, each caught on some run of every C10d draw: *iceberg only* ($T_c$, the worse level ignored: the run $T_c+1$), *public book only* ($\min(\mathit{peak}, T) + q_v$, the hidden quantity ignored: the run $\mathcal{M}$, because $T_c > \mathit{peak}$ by construction), *one tick long* (the run $\mathcal{M}+1$), and for ICEwPI *OQ-11 Interpretation A* ($T + q_v$, the run $\mathcal{M}+1$, because $r < n$). A wrong fill order (the visible level before the iceberg's better hidden slices, or after a slice it outranks) is caught by the (Q) row sequence on the runs $T_c+1$ and $\mathcal{M}$; the classes *standard*, *ICEwPI interleaved* and *ICEwPI behind* make both directions appear.

## 2. The rule, quoted

The governing passages, quoted **verbatim** (byte-copied) and checked against the document each names at run time (`notebooks.certification.cert_support.check_quotes`). A quote that has drifted is flagged **DRIFT** below and recorded as `quote_drift`: the notebook still runs to the end and ends `NOT CERTIFIED`. The regulatory sources are shown as citations, not machine-checked. (Not §4.2's *"cannot combine with IOC or FOK behavior"*: that sentence is about an iceberg's own restriction, not about what a FOK sees, and it is deliberately not cited for the visibility claim.)

How the quotes ground the expectation of section 5:

- **The dichotomy (§4.5 FOK row; §5.1; OQ-11 resolution item 4).** Full fill or nothing; never rests; an unmatched FOK is accepted and cancelled.
- **What the pre-check counts (§4.2.1 Rule 1, in full).** Standard icebergs: all remaining quantity if the price crosses (C10b). ICEwPI: only the slices whose drifted price crosses (C10c), with the worked example. OQ-11: Interpretation B, not A.
- **How an executed FOK fills (§4.2.1 Rule 1 and Rule 2; §6.1; §6.2 items 2, 3, 6, 7).** Price levels best first; one iceberg's slices one after another within the submission; every trade at its maker's (slice's) price, never capped at the FOK limit.
- **The ladder (§4.2; §13.6).** Slice sizes and drifted prices; sign-valid increments; the draw range as an empirical envelope.
- **The permitted between-aggressor state (§4.2.1 Scope clarification; OQ-20).** Why the final book can be one of two (finding F6).
- **The counterparty sequence (review D3-08).** Why every trade row is pinned by order id.
- **The fill order against an iceberg and a worse level (§4.2.1 Rule 2 *Re-ranking*; §6.1; §6.2 item 6).** C10d: each regenerated slice is a new entry at its own price; the FOK always trades the best-ranked counter bid next (finding F15).

In [ ]:
import time

T0 = time.time()   # the measured wall clock (results: wall_clock_s) starts here

from IPython.display import Markdown, display

from notebooks.certification.cert_support import (
    Certification, Citation, SpecQuote, check_quotes, render_quotes,
)

QUOTES = (
    # --- §4.2.1 Rule 1: the load-bearing rule, quoted in full (consecutive parts) ---
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "**Rule 1 — FOK visibility: liquidity check with ICEwPI slice-by-slice price awareness (M6.3, updated OQ-11).**\nA Fill-or-Kill order's pre-match liquidity check considers the total iceberg quantity for opposite-side icebergs at crossing prices. For **standard icebergs (PPD=0)**, all remaining quantity is matchable if the base price crosses.",
        note='C10b: the pre-check counts the hidden quantity of a standard iceberg. (Rule 1, part 1 of 9; the rule is quoted in full, in consecutive parts)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "For **ICEwPI icebergs (PPD != 0)**, the pre-check must compute how many slices would actually cross at their natural drifted prices and count only that volume as matchable.",
        note='C10c: the pre-check projects the drifted ladder slice by slice and counts only the crossing slices. (part 2 of 9)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "**Standard iceberg example:** FOK BUY 3 MW is accepted against an iceberg with peak=2 and total=7 (PPD=0), because all 7 MW are at the same crossing price.",
        note='A request above the visible peak is executed when the hidden quantity covers it. (part 3 of 9)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "**ICEwPI example:** ICEwPI SELL base=50.00, ppd=+0.50, peak=2, total=10. FOK BUY limit=51.00 for 8 MW. Slices at natural prices: 2@50.00, 2@50.50, 2@51.00 = 6 MW crossing. Slice at 51.50 does NOT cross. Matchable = 6 MW < 8 MW requested. **FOK rejected.** If the FOK requested 6 MW: accepted, fills at natural iceberg prices (50.00, 50.50, 51.00). No price capping occurs.",
        note='The worked example of the C10c formula: matchable = min(total, peak * #crossing slices) = min(10, 2 * 3) = 6 MW. (part 4 of 9)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "When accepted, the FOK uses multi-slice matching to fill the full FOK quantity across iceberg slice regenerations within a single `submit_order` call. Each slice trades at the iceberg's (drifted) limit price per the price-determination rule (§5.1.2).",
        note='On execution: every trade row is written by the FOK submission, each at its slice price. (part 5 of 9)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "*Real market rule:* FOK orders see the iceberg's full remaining quantity for matching purposes (Continuous Trading Matching Algorithm §3.3: \"full quantity is exposed to the market for matching, and which may come with a range of limit prices\").",
        note="The card's named quote: FOK orders see the iceberg's full remaining quantity. (part 6 of 9)",
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "For ICEwPI, §5.2.1 states slices \"participate with each slice individually... until either all slices are fully executed or no further execution is possible anymore.\" The price-determination rule (§5.1.2) requires trade price to be set by the iceberg's limit price, not the incoming order's. **See OQ-11 for full analysis.**",
        note='The projection and the price-setter, as the real-market grounding of Rule 1. (part 7 of 9)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "*Implementation:* `_matchable_volume()` must compute slice-by-slice projected prices for ICEwPI icebergs, summing only slices whose projected price crosses the FOK limit. Standard icebergs (PPD=0) sum total remaining quantity as before.",
        note='Quoted for completeness (part 8 of 9); the function names are the rule text, not something this notebook reads (independence rule 1).',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "`_fok_multi_slice_fill()` iterates: dispatch limit order, regenerate iceberg slices at natural drifted prices, re-dispatch remaining quantity, until the FOK is fully filled. No price capping is needed.\n- *Classification:* **Consistent with market rules** (updated from simulator design decision in M6.3; ICEwPI slice-awareness added per OQ-11).",
        note='(part 9 of 9) The classification: consistent with market rules.',
    ),
    # --- §4.2.1 Rule 2: multi-slice matching per aggressive order ---
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        '**Rule 2 — Multi-slice matching per aggressive order (NON, IOC, FOK; M6.4 update).**\nA single aggressive order (NON, IOC, or FOK) can fill multiple iceberg slices within one match cycle. After the visible peak is consumed, the iceberg regenerates a new slice (at its natural drifted price for ICEwPI)',
        note='An executed FOK fills slice after slice of one iceberg in its own submission. (part 1 of 2)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "and matching continues against the same aggressive order until either the aggressive order is fully filled, the iceberg is fully exhausted, or the regenerated slice's price no longer crosses the aggressive order's limit. All slice interactions occur within a single `submit_order` call.",
        note='(part 2 of 2) For an executed FOK the stop is the first: the FOK is fully filled.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "Subject to §6.1, each regenerated slice is ranked as a new entry at its own (drifted) price. The aggressor always trades the best-ranked counter bid in the book next. That may be a different resting order when the drifted slice price is worse than that order's price, or when the slice price equals it and the other order has the earlier timestamp.",
        note='C10d: the crossing slices and the worse visible level are taken in rank order; a slice tying the visible price comes after it (findings F14, F15). (Rule 2 Re-ranking, v2.14, part 1 of 2)',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "The aggressor never trades at a worse price level while a better-priced counter bid can still trade, and that includes the next slice of a better-priced iceberg.",
        note='(part 2 of 2) C10d: no trade on the visible level while a better-priced slice of the iceberg can still trade.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2.1',
        "an iceberg whose visible peak was fully consumed by a prior aggressor's matching process and whose next slice has been registered for later regeneration is a permitted between-aggressor state under the matching algorithm (a new slice is activated when the previous one's quantity reaches zero or when subsequent matching draws against the remaining hidden quantity — see",
        note='Why the final book after an executed FOK that ends exactly on a slice boundary is one of two listed books (finding F6).',
    ),
    # --- §4.2: the iceberg and its ladder ---
    SpecQuote(
        'market-spec.md',
        '§4.2',
        'A regular limit order with hidden total quantity and visible peak quantity.',
        note='The iceberg: a total quantity of which only a peak is shown.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2',
        'When a slice is exhausted, a new slice of peak size is activated and receives a new timestamp.',
        note='The slice sizes: peak, ..., peak, and the remainder.',
    ),
    SpecQuote(
        'market-spec.md',
        '§4.2',
        '**ICEwPI (Iceberg with Price Increment):** Each new slice is priced at the previous slice price ± a fixed delta. Buy icebergs: negative delta (price decreases per slice). Sell icebergs: positive delta (price increases per slice).',
        note='The projected ladder of C10c: slice k at base + k*ppd (sign-valid draws only).',
    ),
    # --- §4.5: the FOK row ---
    SpecQuote(
        'market-spec.md',
        '§4.5',
        '| FOK | No (full or cancel) | No (cancelled) | Regular, Linked basket |',
        note='The M4 dichotomy: the FOK fills its full quantity or nothing, and never rests.',
    ),
    # --- §5.1: the cancellation branch ---
    SpecQuote(
        'market-spec.md',
        '§5.1',
        'CANCELLED (IOC/FOK unmatched; GTD expired; GCT reached; halt/hibernation)',
        note='A killed FOK is an accepted order whose whole quantity is cancelled (not a validation rejection): its quantity goes to the fok_kill bucket.',
    ),
    # --- §6.1 / §6.2: priority, crossing, maker pricing, iceberg price-setter ---
    SpecQuote(
        'market-spec.md',
        '§6.1',
        '**Buy side:** ranked highest price first; ties broken by earliest timestamp.',
        note='The order in which a SELL FOK meets a BUY ladder.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.1',
        '**Sell side:** ranked lowest price first; ties broken by earliest timestamp.',
        note='The order in which a BUY FOK meets a SELL ladder.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 2',
        'Match is possible if: orders are opposite sides, same contract, and `sell_price ≤ buy_price`.',
        note='The crossing test of the pre-check, level by level and slice by slice.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 3',
        "Trade executes at the **resting order's limit price** (maker pricing).",
        note='C10a: each level trades at its own price, never at the FOK limit.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 6',
        'If matching exhausts one price level, continues to next best price level.',
        note='C10a: an executed FOK walks the visible ladder.',
    ),
    SpecQuote(
        'market-spec.md',
        '§6.2 item 7',
        "Iceberg: when a single incoming order matches multiple slices of the same iceberg, the iceberg is always the price-setter.",
        note='C10b/C10c: every slice trades at its own (drifted) price, never at the FOK limit: no price capping.',
    ),
    # --- §13.6: the sign rule (C10c draws are sign-valid) ---
    SpecQuote(
        'market-spec.md',
        '§13.6',
        '**Sign rule (hard).** ICEwPI buy orders must have `PPD < 0`; ICEwPI sell orders must have `PPD > 0`.',
        note='Every C10c draw is sign-valid, checked from its tape (section 4); C09 records the conformance gap.',
    ),
    SpecQuote(
        'market-spec.md',
        '§13.6',
        'Do not encode a fixed numeric magnitude envelope for `PPD` in simulator validation or tests. Any numeric envelope used for robustness/sanity checking must be justified as an empirical choice tied to a specific dataset, not cited as a market rule.',
        note="Why the C10c draw range (|ppd| 1-29 ticks) is stated as the generator's empirical test envelope, never a bound.",
    ),
    # --- open questions and the domain review ---
    SpecQuote(
        'open-questions.md',
        'OQ-11',
        "*Interpretation B — Slice-by-slice crossing, natural prices:* The pre-check computes the number of slices whose natural drifted price actually crosses the FOK limit. Only that quantity is counted as matchable. If the FOK requests more than the crossing quantity, it is rejected. If accepted, all slices trade at their natural (drifted) iceberg prices. No price capping occurs.",
        block='Two interpretations considered:',
        note='The reading C10c certifies.',
    ),
    SpecQuote(
        'open-questions.md',
        'OQ-11',
        "*Interpretation A — Total quantity, price capping:* The pre-check counts all remaining ICEwPI quantity as matchable if the base price crosses. During filling, slices whose natural drifted price exceeds the FOK limit are capped at the FOK limit price.",
        block='Two interpretations considered:',
        note='The rejected reading. Every C10c draw discriminates it (the request matchable + 1 is killed under B and executed under A).',
    ),
    SpecQuote(
        'open-questions.md',
        'OQ-11',
        '**Resolution:** **Resolved. Interpretation B is correct.**',
        block='Resolution:',
        note='OQ-11 resolved: Interpretation B.',
    ),
    SpecQuote(
        'open-questions.md',
        'OQ-11',
        '**FOK requires full-quantity matchability.** §3.5.3: "executed immediately and with its full quantity or, if the order cannot be matched with its entire quantity, deleted." A FOK requesting more volume than the number of slices that actually cross must be rejected',
        block='Resolution:',
        note='The FOK definition the dichotomy rests on.',
    ),
    SpecQuote(
        'open-questions.md',
        'OQ-20',
        "An active iceberg slot whose `order_id` is absent from the LOB at rest between aggressors — i.e. the visible peak was fully consumed by a prior aggressor's matching process and the next slice has not yet been activated — is a **permitted between-aggressor state** under the matching algorithm.",
        block='Ruling.',
        note='The permitted between-aggressor state (finding F6).',
    ),
    SpecQuote(
        'scenario-ladder-review.md',
        'D3-08',
        "Any scenario that draws two same-price icebergs against one\naggressor (C05's interloper family, C08's multi-slice draw, C09) should pin the exact counterparty\nsequence in its Known result.",
        note='Why every trade row is pinned by buyer and seller order id (good practice here: one maker per price level).',
    ),
)
CITATIONS = (
    Citation(
        'Continuous Trading Matching Algorithm (public description) §3.5.3 — docs/omiecid/corpus/Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_Spain_OMIE/Continuous_Trading_Matching_Algorithm_Public_Description.md:307-312',
        'An order submitted wi th the execution restriction FOK is either executed immediately and with its ful l quantity or, if the order canno t be matched with its entire quantity, deleted without entry in the order book. FOK orders can be matched against multiple existing orders in the order book and in that case, create multiple trades. An FOK order is never displayed in the order book.',
    ),
    Citation(
        'Continuous Trading Matching Algorithm (public description) §3.3 — docs/omiecid/corpus/Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_Spain_OMIE/Continuous_Trading_Matching_Algorithm_Public_Description.md:257-259',
        'Iceberg orders are limit orders which are only visible with part of their total quantity in the market, while their full quantity is exposed to the market for matching, and which may come with a range of limit prices.',
    ),
)

QUOTE_CHECKS = check_quotes(QUOTES)
display(Markdown(render_quotes(QUOTE_CHECKS, CITATIONS)))
print(f"{sum(c.ok for c in QUOTE_CHECKS)} of {len(QUOTE_CHECKS)} quotes match their source")

## 3. Configuration

One visible parameter cell. The headless gate overrides it by injecting a cell right after it (`execute_notebook(..., parameters={...})`); everything derived from it is computed in the next cell.

- **`DRAWS = 500`** valid draws **per part** (the card's default for a random scenario), so 2,000 draws in all. Each part has its own seed range, so a seed names one draw of one part: C10a from `SEED0_A`, C10b from `SEED0_B`, C10c from `SEED0_C`, C10d from `SEED0_D`.
- **Runs per draw** (finding F3): C10a 3, C10b 2 or 3 (the drawn request may coincide with $\mathit{total}$ or $\mathit{total}+1$), C10c 2, C10d 4 (the drawn request is always one of $T_c$, $T_c+1$, $\mathcal{M}$, $\mathcal{M}+1$). The battery is run on every run; the results JSON folds a draw's runs into one battery row per check (a check fails for the draw if it fails on any of its runs, and the draw is discarded if I8 marks any run `invalid`).
- **Discards.** A seed the generator refuses (`CapacityBreach`), a draw any of whose run tapes lies outside the capacity margin before the run, or a draw any of whose runs the battery's I8 marks `invalid` after it, is **discarded and replaced by the next seed of its part**. Each part's loop stops at `DRAWS` valid draws or after `2 × DRAWS` attempts; the discard rate is reported. An I8 discard of a draw that passed the pre-run margin is also a failed assertion (finding F10).
- **`EAGER_DRAWS = 2`.** In each part, the first `EAGER_DRAWS` **valid** draws (the first valid seeds) also run eager (every run of the draw), and each eager record is compared, integer field by integer field, with its jitted record (`compare_records`). On top of these, an eager run is **forced** for the first valid draw of each class below. Every part and every class **must** have its eager comparison: a missing one is a failed `eager_jit` assertion. Eager runs are chosen among valid draws, so a discarded seed cannot remove them.
- **The classes** (each must appear among the valid draws: the coverage assertion of section 7). C10a: resting side (BUY, SELL) × ($L = 1$, $L \ge 2$ levels). C10b: iceberg side × (drawn request $\le \mathit{total}$, drawn request $= \mathit{total} + 1$). C10c: iceberg side × ($r = 1$, $r \ge 2$ crossing slices). C10d: iceberg side × (standard; ICEwPI with the visible level *interleaved* — ranked ahead of at least one crossing slice; ICEwPI with the visible level *behind* every crossing slice). Eighteen classes.
- **The draw envelope is empirical** (finding F11; C10d's is this notebook's own: base 30.00–90.00 EUR/MWh, peak 1.0–5.0 MW, standard $n \in [2, 8]$ with a last slice of 0.1 MW to a full peak, ICEwPI $n \in [3, 10]$ full slices with $|\mathit{ppd}|$ 1–29 ticks and $r \in [2, n-1]$, the visible level 1–20 ticks behind the base (standard) or anywhere strictly behind the base up to the limit (ICEwPI), $q_v$ 1.0–15.0 MW, the standard FOK 0–20 ticks through the visible level). The tick grids come from `OMIEProductConfig()` and serve only to display euros.

In [ ]:
CERT_ID = "C10"            # C00-C17 are the ladder; C90-C99 are reserved for the harness
DRAWS = 500                # valid draws PER PART (card: random scenarios default to 500 draws)
SEED0_A = 0                # first seed of C10a (C10.fok_visible)
SEED0_B = 1_000_000        # first seed of C10b (C10.fok_hidden): a disjoint range, so a seed names one draw
SEED0_C = 2_000_000        # first seed of C10c (C10.fok_icewpi)
SEED0_D = 3_000_000        # first seed of C10d (notebook-local: FOK vs an iceberg plus a worse visible level)
EAGER_DRAWS = 2            # per part, the first EAGER_DRAWS valid draws also run eager (plus the first of each class)

In [ ]:
import pathlib

import numpy as np

from omiecid_simulator.jaxomiecid import omie_tape_synth as synth
from omiecid_simulator.omiecid.omie_config import OMIEProductConfig
from omiecid_simulator.omiecid.order_types import ExecRestriction, Side, Validity

PARTS = ("C10a", "C10b", "C10c", "C10d")
CONSTRUCTION = {"C10a": "C10.fok_visible", "C10b": "C10.fok_hidden", "C10c": "C10.fok_icewpi",
                "C10d": "C10.fok_ice_worse_level"}   # C10d: built in section 4 (finding F14), not a registry key
REGISTERED = tuple(CONSTRUCTION[p] for p in PARTS if CONSTRUCTION[p] in synth.CONSTRUCTIONS)
SEED0 = {"C10a": int(SEED0_A), "C10b": int(SEED0_B), "C10c": int(SEED0_C), "C10d": int(SEED0_D)}
PART_TITLE = {"C10a": "boundary (visible ladder)", "C10b": "hidden volume (standard iceberg)",
              "C10c": "price-incremented (ICEwPI)", "C10d": "iceberg plus a worse visible level"}

CONFIG = OMIEProductConfig()
ENVELOPE = synth.DEFAULT_ENVELOPE
BUY, SELL = int(Side.BUY), int(Side.SELL)
NON, IOC, FOK, ICE = (int(ExecRestriction.NON), int(ExecRestriction.IOC), int(ExecRestriction.FOK),
                      int(ExecRestriction.ICE))
EXEC_NAME = {NON: "NON", IOC: "IOC", FOK: "FOK", ICE: "ICE"}
GFS = int(Validity.GFS)

SLICE_MARGIN = 64                     # capacity-validity precondition: every iceberg has ceil(total/peak) <= 64
PRICE_INT_MAX = 1_999_800             # §2.3: +9,999 EUR/MWh in the offset encoding (0 is -9,999)
QTY_INT_MIN, QTY_INT_MAX = 1, 9_990   # §2.3: 0.1 MW and 999 MW in quantity ticks

CLASSES = {
    "C10a": tuple(f"C10a: ladder {s}, {k}" for s in ("BUY", "SELL") for k in ("1 level", ">= 2 levels")),
    "C10b": tuple(f"C10b: iceberg {s}, drawn {k}" for s in ("BUY", "SELL") for k in ("req <= total", "req = total + 1")),
    "C10c": tuple(f"C10c: iceberg {s}, {k}" for s in ("BUY", "SELL") for k in ("r = 1", "r >= 2")),
    "C10d": tuple(f"C10d: iceberg {s}, {k}" for s in ("BUY", "SELL")
                  for k in ("standard", "ICEwPI, visible interleaved", "ICEwPI, visible behind")),
}
ALL_CLASSES = tuple(c for p in PARTS for c in CLASSES[p])

_own = sorted(pathlib.Path.cwd().glob(f"{CERT_ID}_*.ipynb"))
NOTEBOOK = _own[0].name if len(_own) == 1 else "C10_fok_and_hidden_volume.ipynb"

for _p in PARTS:
    print(f"{_p} ({PART_TITLE[_p]}): {CONSTRUCTION[_p]} x {DRAWS} valid draws from seed {SEED0[_p]} "
          f"(at most {2 * DRAWS} attempts)")
print(f"eager: per part the first {EAGER_DRAWS} valid draws + the first of each of {len(ALL_CLASSES)} classes")
print(f"registered constructions (digested in the results): {REGISTERED}; notebook-local: "
      f"{[CONSTRUCTION[p] for p in PARTS if CONSTRUCTION[p] not in REGISTERED]}")
print(f"price tick {CONFIG.price_tick} EUR/MWh, qty tick {CONFIG.qty_tick} MW (display only)")
print(f"capacity envelope: {ENVELOPE}; slice margin <= {SLICE_MARGIN} per iceberg")

## 4. Dataset

Each construction emits a GFS tape on one contract: order ids $1, 2, \ldots$ and tape timestamps strictly increasing in row order, the FOK last, then pad rows. Write $\sigma$ for the resting side and $\bar\sigma$ for the FOK's.

| part | resting submissions $0..L-1$ | the FOK (submission $L$) |
|---|---|---|
| C10a `C10.fok_visible` | $L \in [1, 5]$ $\sigma$ NON orders, one per level, each strictly worse than the last for a taker, quantities $q_i$; $V = \sum_i q_i$ | $\bar\sigma$ FOK, limit **at the deepest level's price** (every level crosses), request $V + \delta$, $\delta \in \{-1, 0, +1\}$ |
| C10b `C10.fok_hidden` | one $\sigma$ ICE at $p_{ice}$, $\mathit{ppd} = 0$, $0 < \mathit{peak} < \mathit{total}$, $n = \lceil \mathit{total}/\mathit{peak} \rceil \in [2, 14]$ | $\bar\sigma$ FOK, limit **strictly through** $p_{ice}$, request $\mathit{req} \in [\mathit{peak}+1, \mathit{total}]$ or $\mathit{total}+1$ |
| C10c `C10.fok_icewpi` | one $\sigma$ ICE at $\mathit{base}$, sign-valid $\mathit{ppd} \ne 0$, $\mathit{total} = n\cdot\mathit{peak}$, $n \in [3, 14]$ | $\bar\sigma$ FOK, limit **exactly** $\mathit{base} + (r-1)\,\mathit{ppd}$, $r \in [1, n-1]$; request $M = r\cdot\mathit{peak}$ or $M + 1$ |
| C10d `C10.fok_ice_worse_level` (built here) | submission 0: one $\sigma$ ICE at $\mathit{base}$ — standard ($\mathit{ppd} = 0$, $n \in [2, 8]$) or sign-valid ICEwPI ($\mathit{total} = n\cdot\mathit{peak}$, $n \in [3, 10]$); submission 1: one $\sigma$ NON at $p_v$ **strictly worse** than $\mathit{base}$, $q_v$ | $\bar\sigma$ FOK (submission 2): standard — limit $p_v$ or 1–20 ticks through it; ICEwPI — limit between slice $r-1$'s price (inclusive) and slice $r$'s (exclusive), $r \in [2, n-1]$, and $p_v$ strictly behind the base, at or inside the limit; request one of $T_c,\ T_c+1,\ \mathcal{M},\ \mathcal{M}+1$ |

**The twins** (finding F3). `twin(draw, req)` copies the tape, sets the FOK row's quantity to `req` and nothing else, and recomputes the capacity bounds from the new tape (`synth.tape_capacity_bounds`). The run set of each draw is computed from the **tape** by section 5's pre-check projection ($V$, $\mathit{total}$, $M$), never from `draw.expected`.

**What section 5 reads.** The expectation reads the **tape rows** through one reference walk, and `draw.params` for the closed form, which is cross-checked against the walk on every draw and against the tape (the precondition below). `draw.expected` is never read.

**The precondition, checked from the tape for every valid draw** (an aggregate assertion in section 7): one contract, order ids $1..N$ in row order, tape timestamps strictly increasing, all GFS, prices and sizes within §2.3; the resting rows all on one side and the FOK (the last row, no peak, no increment) on the other. C10a: every resting row NON, prices strictly worse in submission order, the FOK limit equal to the deepest price, request $\in \{V-1, V, V+1\}$. C10b: one ICE row, $\mathit{ppd} = 0$, $0 < \mathit{peak} < \mathit{total}$, $2 \le n \le 64$, the FOK strictly through $p_{ice}$, request $> \mathit{peak}$. C10c: one ICE row, sign-valid $\mathit{ppd} \ne 0$, $2 \le n \le 64$, every declared slice price inside §2.3, the crossing slices a non-empty prefix $k < r$ with $r < n$, the FOK limit equal to slice $r-1$'s price. C10d: an ICE row then a plain NON row, both on $\sigma$; $0 < \mathit{peak} < \mathit{total}$, $2 \le n \le 64$; $\mathit{ppd} = 0$ (then every slice crosses) or sign-valid with the crossing slices a prefix of length $r \in [2, n-1]$; every declared slice price inside §2.3; $p_v$ strictly worse than the base and crossing the limit; request $\in \{T_c, T_c+1, \mathcal{M}, \mathcal{M}+1\}$. Every value equal to its `params` entry.

**The capacity margin** (scenario-ladder-plan, *The capacity-validity precondition*, five ceilings), checked from **every run's tape** before the run: the analytic bounds (`draw.capacity_bounds` for the drawn tape, and the generic `tape_capacity_bounds(tape)` for every tape) within the envelope (≤ 80 resting per side, ≤ 80 trades, hop bound ≤ 160, ≤ 800 concurrent icebergs), and the largest $\lceil \mathit{qty}/\mathit{peak} \rceil$ over the tape's iceberg rows ≤ 64. A tape outside it is never run. After the run, the battery's I8 confirms the flags.

Below: the first draw of each part in readable units beside the integers, the book shapes, the class counts and the envelope actually drawn in the initial seed ranges. C10d's builder (`build_c10d`) is in the code cell below; a draw outside the capacity envelope raises `CapacityBreach`, which is treated exactly as a generator refusal.

In [ ]:
import dataclasses
import types
from collections import Counter

from omiecid_simulator.omiecid.encoding import decode_price, decode_qty, encode_price

C_CONTRACT, C_SIDE, C_PRICE, C_QTY, C_OID = (synth.COL_CONTRACT, synth.COL_SIDE, synth.COL_PRICE, synth.COL_QTY,
                                             synth.COL_OID)
C_AGENT, C_TS, C_EXEC, C_VALIDITY, C_PEAK, C_PPD = (synth.COL_AGENT, synth.COL_TS, synth.COL_EXEC,
                                                    synth.COL_VALIDITY, synth.COL_PEAK, synth.COL_PPD)


def real_rows(tape):
    # The real rows (order_id >= 1), in tape (= submission) order.
    t = np.asarray(tape)
    return t[t[:, C_OID] >= 1]


def side_name(side):
    return "BUY" if int(side) == BUY else "SELL"


def other(side):
    return SELL if int(side) == BUY else BUY


def crosses(rest_side, p_rest, p_in):
    # §6.2 item 2: sell price <= buy price, for a resting entry on rest_side and an incoming limit p_in.
    return int(p_rest) <= int(p_in) if int(rest_side) == SELL else int(p_rest) >= int(p_in)


def ceil_div(a, b):
    return -(-int(a) // int(b))


def sign_valid(side, ppd):
    # §4.2 / §13.6: BUY icebergs negative increments, SELL icebergs positive.
    return int(ppd) < 0 if int(side) == BUY else int(ppd) > 0


def max_slices(tape):
    # Largest ceil(qty / peak) over the tape's iceberg rows (0 when there are none).
    ice = [r for r in real_rows(tape) if int(r[C_EXEC]) == ICE]
    return max((ceil_div(r[C_QTY], max(1, int(r[C_PEAK]))) for r in ice), default=0)


def fok_index(tape):
    # Submission index of the FOK row (the last real row in every C10 construction).
    rows = real_rows(tape)
    idx = [i for i, r in enumerate(rows) if int(r[C_EXEC]) == FOK]
    return idx[-1] if idx else None


def twin(draw, req):
    # The draw's tape with ONLY the FOK row's quantity set to req (finding F3); capacity bounds recomputed.
    rows = real_rows(draw.tape)
    s = fok_index(draw.tape)
    if s is None:
        raise ValueError("no FOK row")
    if int(rows[s][C_QTY]) == int(req):
        return draw
    t = np.array(draw.tape, dtype=np.int32, copy=True)
    t[np.where(t[:, C_OID] == int(rows[s][C_OID]))[0][0], C_QTY] = int(req)
    t = np.ascontiguousarray(t)
    cb = synth.tape_capacity_bounds(t)
    return dataclasses.replace(
        draw, tape=t, params={**draw.params, "req_int": int(req), "twin_of_req_int": int(rows[s][C_QTY])},
        expected={}, hop_bound=int(cb["hops"]), capacity_bounds=types.MappingProxyType(dict(cb)),
        meta={**draw.meta, "twin": True})


DRAWN = {}       # (part, seed) -> Draw, or None when the generator refused the seed
REFUSED = {}     # (part, seed) -> the generator's CapacityBreach message


def build_c10d(seed):
    # C10d (finding F14), built from the rules: an iceberg, a strictly worse visible NON level behind it on the
    # same side, then a FOK on the other side whose limit reaches that level. Integer ticks throughout.
    rng = np.random.default_rng([10, 4, int(seed)])
    rs = BUY if int(rng.integers(2)) == 0 else SELL
    d = 1 if rs == SELL else -1                         # one tick worse for a taker of side rs
    ladder = "standard" if int(rng.integers(2)) == 0 else "icewpi"
    base = int(encode_price(30.0, CONFIG)) + int(rng.integers(0, 6001))      # 30.00..90.00 EUR/MWh
    pk = int(rng.integers(10, 51))                                            # 1.0..5.0 MW
    if ladder == "standard":
        n = int(rng.integers(2, 9))
        T = (n - 1) * pk + int(rng.integers(1, pk + 1))                       # a short last slice allowed
        ppd, r = 0, n                                                         # every slice at the base
        pv = base + d * int(rng.integers(1, 21))                              # 1..20 ticks behind the base
        lim = pv + d * int(rng.integers(0, 21))                               # at or through the visible level
    else:
        n = int(rng.integers(3, 11))
        T = n * pk
        ppd = d * int(rng.integers(1, 30))                                    # sign-valid (§13.6)
        r = int(rng.integers(2, n))                                           # 2..n-1 crossing slices
        lim = base + (r - 1) * ppd + d * int(rng.integers(0, abs(ppd)))       # slice r-1 crosses, slice r does not
        pv = base + d * int(rng.integers(1, abs(lim - base) + 1))             # strictly behind the base, crosses lim
    qv = int(rng.integers(10, 151))                                           # 1.0..15.0 MW
    tc = min(T, pk * r)
    m = tc + qv
    req = int(rng.choice([tc, tc + 1, m, m + 1]))
    tb = synth.TapeBuilder(CONFIG)
    o_i = tb.add(side=rs, price_int=base, qty_int=T, peak_int=pk, ppd_int=ppd, exec_restriction=ICE)
    o_v = tb.add(side=rs, price_int=pv, qty_int=qv, exec_restriction=NON)
    o_f = tb.add(side=other(rs), price_int=lim, qty_int=req, exec_restriction=FOK)
    tape = tb.build()
    cb = synth.tape_capacity_bounds(tape)
    synth.check_envelope(ENVELOPE, **cb)                                      # CapacityBreach = a refusal
    params = {"rest_side": rs, "fok_oid": o_f, "req_int": req, "p_fok_int": lim, "ladder": ladder,
              "ice_oid": o_i, "base_int": base, "ppd_int": ppd, "peak_int": pk, "total_int": T, "n_slices": n,
              "reachable_slices": r, "vis_oid": o_v, "vis_price_int": pv, "vis_qty_int": qv}
    return synth.Draw(construction=CONSTRUCTION["C10d"], seed=int(seed), tape=tape, params=params, expected={},
                      hop_bound=int(cb["hops"]),
                      capacity_bounds=types.MappingProxyType({a: int(cb[a]) for a in synth.AXES}),
                      oracle_kind="exact-parametric", meta={"builder": "C10 notebook section 4 (finding F14)"})


def prepare(part, seed):
    # Draw one seed of one part (the run loop of section 7 calls this for a re-draw too).
    try:
        d = build_c10d(seed) if part == "C10d" else synth.CONSTRUCTIONS[CONSTRUCTION[part]](seed)
    except synth.CapacityBreach as exc:
        d = None
        REFUSED[(part, seed)] = str(exc)
    DRAWN[(part, seed)] = d
    return d


for _p in PARTS:
    for _s in range(SEED0[_p], SEED0[_p] + DRAWS):
        prepare(_p, _s)


def tape_table(draw, title):
    rows = ["| s | oid | side | exec | price (EUR/MWh) | qty (MW) | peak (MW) | ppd (EUR/MWh) | price_int | qty_int | "
            "peak_int | ppd_int | agent | tape ts |", "|---|---|---|---|---|---|---|---|---|---|---|---|---|---|"]
    for k, r in enumerate(real_rows(draw.tape)):
        rows.append(f"| {k} | {int(r[C_OID])} | {side_name(r[C_SIDE])} | {EXEC_NAME.get(int(r[C_EXEC]), int(r[C_EXEC]))} | "
                    f"{decode_price(int(r[C_PRICE]), CONFIG):.2f} | {decode_qty(int(r[C_QTY]), CONFIG):.1f} | "
                    f"{decode_qty(int(r[C_PEAK]), CONFIG):.1f} | {int(r[C_PPD]) * CONFIG.price_tick:+.2f} | "
                    f"{int(r[C_PRICE])} | {int(r[C_QTY])} | {int(r[C_PEAK])} | {int(r[C_PPD])} | "
                    f"{int(r[C_AGENT])} | {int(r[C_TS])} |")
    return f"**{title}**\n\n" + "\n".join(rows)


FIRST = {p: next((s for s in range(SEED0[p], SEED0[p] + DRAWS) if DRAWN[(p, s)] is not None), None) for p in PARTS}
for _p in PARTS:
    if FIRST[_p] is None:
        print(f"{_p}: every seed of the initial range was refused by the generator")
        continue
    _d = DRAWN[(_p, FIRST[_p])]
    display(Markdown(tape_table(_d, f"{_p} — {CONSTRUCTION[_p]}: seed {FIRST[_p]} (digest `{_d.digest()}`)")))
    print(f"seed {FIRST[_p]} params: {_d.params}")
    print(f"seed {FIRST[_p]} capacity bounds: {dict(_d.capacity_bounds)}; largest ceil(qty/peak): {max_slices(_d.tape)}")
print(f"seeds refused by the generator in the initial ranges: {len(REFUSED)}")


def crossing_slices(ice, limit):
    # #{k >= 0 : slice k's drifted price crosses limit}, capped at the slice count (a prefix: finding F4).
    n = ceil_div(ice[C_QTY], ice[C_PEAK])
    r = 0
    while r < n and crosses(ice[C_SIDE], int(ice[C_PRICE]) + r * int(ice[C_PPD]), limit):
        r += 1
    return r


def class_of(part, draw):
    # The class of a draw, read from its tape (section 3).
    rows = real_rows(draw.tape)
    rest, fok = rows[0], rows[-1]
    side = side_name(rest[C_SIDE])
    if part == "C10a":
        return f"C10a: ladder {side}, " + ("1 level" if len(rows) == 2 else ">= 2 levels")
    if part == "C10b":
        return f"C10b: iceberg {side}, drawn " + ("req = total + 1" if int(fok[C_QTY]) > int(rest[C_QTY]) else "req <= total")
    if part == "C10c":
        return f"C10c: iceberg {side}, " + ("r = 1" if crossing_slices(rest, fok[C_PRICE]) == 1 else "r >= 2")
    if int(rest[C_PPD]) == 0:
        return f"C10d: iceberg {side}, standard"
    vis = rows[1]
    last = int(rest[C_PRICE]) + (crossing_slices(rest, fok[C_PRICE]) - 1) * int(rest[C_PPD])
    behind = int(vis[C_PRICE]) > last if int(rest[C_SIDE]) == SELL else int(vis[C_PRICE]) < last
    return f"C10d: iceberg {side}, ICEwPI, visible " + ("behind" if behind else "interleaved")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))
for ax, _p in zip(axes, PARTS):
    if FIRST[_p] is None:
        ax.set_title(f"{_p}: no draw")
        continue
    _d = DRAWN[(_p, FIRST[_p])]
    _rows = real_rows(_d.tape)
    _f = _rows[fok_index(_d.tape)]
    for r in _rows[:-1]:
        _sgn = 1 if int(r[C_SIDE]) == BUY else -1
        if int(r[C_EXEC]) == ICE:
            _n = ceil_div(r[C_QTY], r[C_PEAK])
            for k in range(_n):
                p_k = int(r[C_PRICE]) + k * int(r[C_PPD])
                x_k = min(int(r[C_PEAK]), int(r[C_QTY]) - k * int(r[C_PEAK]))
                hit = crosses(r[C_SIDE], p_k, _f[C_PRICE])
                y = decode_price(p_k, CONFIG)
                left = sum(min(int(r[C_PEAK]), int(r[C_QTY]) - j * int(r[C_PEAK])) for j in range(k)) if int(r[C_PPD]) == 0 else 0
                ax.barh(y, _sgn * decode_qty(x_k, CONFIG), left=_sgn * decode_qty(left, CONFIG),
                        height=max(abs(int(r[C_PPD])) * 0.006, 0.08), edgecolor="black", lw=0.3,
                        color="tab:green" if _sgn > 0 else "tab:red", alpha=0.9 if hit else 0.25,
                        hatch=None if hit else "//")
        else:
            ax.barh(decode_price(int(r[C_PRICE]), CONFIG), _sgn * decode_qty(int(r[C_QTY]), CONFIG), height=0.08,
                    color="tab:green" if _sgn > 0 else "tab:red", edgecolor="black", lw=0.3)
    ax.axhline(decode_price(int(_f[C_PRICE]), CONFIG), color="black", ls="--", lw=0.9,
               label=f"FOK {side_name(_f[C_SIDE])} limit {decode_price(int(_f[C_PRICE]), CONFIG):.2f}, "
                     f"req {decode_qty(int(_f[C_QTY]), CONFIG):.1f} MW")
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("MW: buy right, sell left (stacked: the iceberg's slices)", fontsize=7)
    ax.set_ylabel("price (EUR/MWh)", fontsize=7)
    ax.set_title(f"{_p}, seed {FIRST[_p]}: the resting side and the FOK limit", fontsize=8)
    ax.legend(loc="best", fontsize=6)
plt.tight_layout()
plt.show()

_class_counts = {}
for _p in PARTS:
    _main = [d for (q, s), d in DRAWN.items() if q == _p and d is not None]
    _c = Counter(class_of(_p, d) for d in _main)
    _class_counts.update({k: _c.get(k, 0) for k in CLASSES[_p]})
display(Markdown("**Classes in the initial seed ranges** (counts)\n\n| class | draws |\n|---|---|\n"
                 + "\n".join(f"| {k} | {v} |" for k, v in _class_counts.items())))

_env = {}
for _p in PARTS:
    _main = [d for (q, s), d in DRAWN.items() if q == _p and d is not None]
    if not _main:
        continue
    _rs = [real_rows(d.tape) for d in _main]
    _env[f"{_p} resting orders"] = [len(r) - 1 for r in _rs]
    _env[f"{_p} resting qty per order (MW)"] = [decode_qty(int(x[C_QTY]), CONFIG) for r in _rs for x in r[:-1]]
    _env[f"{_p} FOK request (MW)"] = [decode_qty(int(r[-1][C_QTY]), CONFIG) for r in _rs]
    if _p in ("C10b", "C10c"):
        _env[f"{_p} peak (MW)"] = [decode_qty(int(r[0][C_PEAK]), CONFIG) for r in _rs]
        _env[f"{_p} n = ceil(total/peak)"] = [max_slices(d.tape) for d in _main]
    if _p == "C10b":
        _env["C10b FOK limit through p_ice (ticks)"] = [abs(int(r[-1][C_PRICE]) - int(r[0][C_PRICE])) for r in _rs]
    if _p == "C10c":
        _env["C10c |ppd| (ticks)"] = [abs(int(r[0][C_PPD])) for r in _rs]
        _env["C10c r (crossing slices)"] = [crossing_slices(r[0], r[-1][C_PRICE]) for r in _rs]
    if _p == "C10d":
        _env["C10d peak (MW)"] = [decode_qty(int(r[0][C_PEAK]), CONFIG) for r in _rs]
        _env["C10d n = ceil(total/peak)"] = [max_slices(d.tape) for d in _main]
        _env["C10d |ppd| (ticks; 0 = standard)"] = [abs(int(r[0][C_PPD])) for r in _rs]
        _env["C10d r (crossing slices)"] = [crossing_slices(r[0], r[-1][C_PRICE]) for r in _rs]
        _env["C10d visible level behind the base (ticks)"] = [abs(int(r[1][C_PRICE]) - int(r[0][C_PRICE])) for r in _rs]
        _env["C10d visible qty (MW)"] = [decode_qty(int(r[1][C_QTY]), CONFIG) for r in _rs]
display(Markdown("**The envelope actually drawn** (the generator's empirical test ranges, never bounds: finding F11)\n\n"
                 "| quantity | min | max |\n|---|---|---|\n"
                 + "\n".join(f"| {k} | {min(v):g} | {max(v):g} |" for k, v in _env.items())))

## 5. Known result

**Notation.** Integer ticks throughout. Every resting order rests on side $\sigma$; the FOK $o_F$ (side $\bar\sigma$, limit $\ell$, request $\mathit{req}$) is the last submission, number $s_F$. A resting entry *crosses* $\ell$ iff sell price ≤ buy price (§6.2 item 2). A trade row is pinned as $[\,\mathit{buyer},\ \mathit{seller},\ \mathit{price},\ \mathit{qty},\ s\,]$: the buyer's and the seller's order ids, the price and quantity in ticks, and the submission that wrote it (finding F7). An iceberg (base $b$, increment $\mathit{ppd}$, peak $\pi$, total $T$) has slices $k = 0, \ldots, n-1$, $n = \lceil T/\pi \rceil$:

$$p_k = b + k\,\mathit{ppd}, \qquad x_k = \min\big(\pi,\ T - k\,\pi\big)$$

(§4.2; $\mathit{ppd} = 0$ for C10b).

**The pre-check** (§4.2.1 Rule 1; OQ-11 Interpretation B). Before matching, the FOK counts the **matchable volume** $\mathcal{M}$ of the book on the other side: every visible order that crosses $\ell$ counts its quantity; every iceberg counts its current slice and its projected later slices, each at its own drifted price, **as long as** they cross $\ell$:

$$\mathcal{M} = \sum_{\text{visible } j \text{ crossing } \ell} q_j \;+\; \sum_{\text{icebergs}} \ \sum_{k \text{ crossing } \ell} x_k .$$

**(V) The dichotomy** (§4.5; §5.1; M4). If $\mathit{req} \le \mathcal{M}$ the FOK is **executed**: it matches exactly as an incoming order would (below) and fills exactly $\mathit{req}$. Otherwise it is **killed**: no trade, nothing changes in the book, and the FOK's whole quantity is cancelled (`fok_kill`). Either way the FOK never rests, and every order is accepted (finding F2).

**(W) The reference walk** (§4.2, §4.2.1 Rules 1–2, §6.1, §6.2 items 2–4, 6, 7). Rows are processed in submission order. An incoming order meets the opposite side ranked by price, then entry (§6.1); while it has quantity and the best opposite entry crosses, it trades $f = \min(\text{left}, \text{visible})$ **at the resting entry's price** (for an iceberg, its current slice price $p_k$: §6.2 item 7 — never the FOK limit). A resting slice used up is replaced at once by the iceberg's next slice, at $p_{k+1}$, which the same incoming order keeps matching if it crosses (Rule 2). What is left of a NON or ICE order rests at its limit (§6.2 item 4); a FOK is first passed through the pre-check above. It is C09's walk, with the pre-check added.

**The closed forms** (from `draw.params`; checked against (W) on every draw):

- **C10a.** $V = \sum_{i<L} q_i$ (every level crosses: $\ell$ is the deepest price). Run set $\{V-1, V, V+1\}$: executed iff $\mathit{req} \le V$. On execution the FOK walks the levels in submission order (each strictly worse), level $i$ giving $f_i = \min(q_i,\ \mathit{req} - \sum_{j<i} f_j)$ at **its own price** $p_i$; $\mathit{req} = V - 1$ leaves exactly 1 tick resting on the deepest level.
- **C10b.** $\mathcal{M} = T$ — the hidden quantity counts. Run set $\{\mathit{req}, T, T+1\}$: executed iff $\mathit{req} \le T$; on execution the trades are the slices $x_0, x_1, \ldots$ taken in order until $\mathit{req}$ is met (the last one possibly in part), **every one at $p_{ice}$**, not at the FOK limit; the iceberg keeps $T - \mathit{req}$ (visible + hidden).
- **C10c.** $r = \#\{k \ge 0 : p_k \text{ crosses } \ell\}$ (SELL iceberg: $p_k \le \ell$; BUY iceberg: $p_k \ge \ell$; finding F4) and

$$M = \min\big(T,\ \pi \cdot r\big).$$

  Run set $\{M, M+1\}$: executed iff $\mathit{req} \le M$. On execution the trades are $[\,b_o, s_o, p_k, x_k, s_F\,]$ for $k = 0..r-1$ — the drifted prices $b, b+\mathit{ppd}, \ldots, b+(r-1)\,\mathit{ppd}$, uncapped (C09's price sequence, summed into $M$); the iceberg keeps $T - M$.
- **C10d** (findings F14–F16). The iceberg's crossing slices, $r = \#\{k \ge 0 : p_k \text{ crosses } \ell\}$ ($r = n$ when $\mathit{ppd} = 0$), give $T_c = \min(T,\ \pi r)$; the visible level crosses by construction, so

$$\mathcal{M} = T_c + q_v .$$

  Run set $\{T_c,\ T_c+1,\ \mathcal{M},\ \mathcal{M}+1\}$: executed iff $\mathit{req} \le \mathcal{M}$. On execution the FOK takes the **pieces** — slice $k < r$ ($x_k$ at $p_k$, maker the iceberg) and the visible level ($q_v$ at $p_v$, maker the NON) — in rank order (§6.1; §4.2.1 Rule 2 *Re-ranking*): better price first; at equal price the earlier entry, where slice 0 entered before the visible order and every slice $k \ge 1$ after it (finding F15). Each piece gives $\min(\text{its size}, \text{what is left})$ at **its own price** until $\mathit{req}$ is met. So for a standard iceberg the request $T_c$ takes the whole iceberg and nothing from the visible level, and $T_c + 1$ takes the whole iceberg and exactly 1 tick from it; for an *interleaved* ICEwPI the visible level trades between two slices. The iceberg keeps $T - (\text{its fills})$, the visible order $q_v - (\text{its fill})$.

**(B) The final book.** On a kill: the book as it was before the FOK (one book). On an execution: the walk's book; if the FOK ended exactly on an iceberg's slice boundary with hidden quantity left, **one of** two books — the next slice shown at its own price, or not yet shown (finding F6).

**(R) Rollback** (M4). On a kill, the snapshot before the FOK's submission and the snapshot after it are structurally equal (finding F5); separately, the battery's M4 holds on every run.

**Discrimination** (findings F9, F16). Per draw, each named wrong pre-check must give a different verdict from (V) on at least one run of the draw; it is asserted from the tape alone (no engine).

**What is asserted, per valid draw** (one combined assertion; expected and observed recorded in full as canonical compact JSON): for every run of the draw, keyed by its request — the verdict; accepted and rejected ids; the trade rows in order, field for field; filled and remaining (visible + hidden) per order id; cancellations per order id and cause; evictions (expected none); the final book `[price_int, qty_int, oid]` per side, one of the listed books; and on a kill, the rollback equality. Separately: the dataset precondition, the closed forms against (W), the discrimination, M4, the battery, I3 and I5 (each failing only on `status == "fail"`, with the `not_applicable` counts reported: finding F13), the capacity checks, coverage and the eager comparisons (section 7).

The code below implements (W) and the pre-check as functions of a tape's rows, and the closed forms as functions of `draw.params`. None of them reads `draw.expected`, and none runs the engine.

In [ ]:
import json


def canon(x):
    # Canonical compact JSON: how every per-draw expected / observed value is recorded (in full).
    return json.dumps(x, sort_keys=True, separators=(",", ":"))


def rank_key(side, price, entry):
    # §6.1: SELL lowest price first, BUY highest price first; ties by the earlier entry.
    return ((int(price) if int(side) == SELL else -int(price)), int(entry))


def projected_volume(e, limit):
    # §4.2.1 Rule 1: what one resting entry contributes to a FOK's pre-check at `limit`: its current
    # visible quantity if it crosses, plus, for an iceberg, every later slice at its own drifted price
    # while that price crosses (the prefix: finding F4).
    vol = e["vis"] if crosses(e["side"], e["price"], limit) else 0
    if e["ice"] is not None and vol > 0:
        ic, hidden, k = e["ice"], e["hidden"], e["ice"]["k"]
        while hidden > 0:
            k += 1
            x = min(ic["peak"], hidden)
            if not crosses(e["side"], ic["base"] + k * ic["ppd"], limit):
                break
            vol += x
            hidden -= x
    return vol


def walk(tape):
    # (W): the reference walk over the real rows, with the FOK pre-check.
    # Returns (trades, final book entries, FOK outcome {oid: (verdict, matchable)}, the book just before the FOK).
    book, trades, seq, fok_out, before_fok = [], [], [0], {}, None

    def nxt():
        seq[0] += 1
        return seq[0]

    def activate(e):
        # The iceberg's next slice: a new entry at the next drifted price (§4.2).
        ic = e["ice"]
        ic["k"] += 1
        e["vis"] = min(ic["peak"], e["hidden"])
        e["hidden"] -= e["vis"]
        e["price"] = ic["base"] + ic["k"] * ic["ppd"]
        e["seq"] = nxt()

    for s, r in enumerate(real_rows(tape)):
        side, p, q, oid, ex = int(r[C_SIDE]), int(r[C_PRICE]), int(r[C_QTY]), int(r[C_OID]), int(r[C_EXEC])
        ice = ex == ICE
        peak = int(r[C_PEAK]) if ice else q
        vis = min(peak, q)
        inc = {"side": side, "price": p, "vis": vis, "hidden": q - vis, "oid": oid, "seq": nxt(), "fresh": False,
               "ice": {"base": p, "ppd": int(r[C_PPD]), "peak": peak, "k": 0} if ice else None}
        if ex == FOK:
            before_fok = [dict(e, ice=None if e["ice"] is None else dict(e["ice"])) for e in book]
            m = sum(projected_volume(e, p) for e in book if e["side"] != side)
            if q > m:
                fok_out[oid] = ("killed", m)        # §4.5 / §5.1: nothing trades, nothing changes
                continue
            fok_out[oid] = ("executed", m)
        while True:
            while inc["vis"] > 0:
                opp = [e for e in book if e["side"] != side and e["vis"] > 0 and crosses(e["side"], e["price"], inc["price"])]
                if not opp:
                    break
                best = min(opp, key=lambda e: rank_key(e["side"], e["price"], e["seq"]))
                f = min(inc["vis"], best["vis"])
                buyer, seller = (oid, best["oid"]) if side == BUY else (best["oid"], oid)
                trades.append([buyer, seller, best["price"], f, s])   # at the resting entry's (slice's) price
                inc["vis"] -= f
                best["vis"] -= f
                best["fresh"] = False
                if best["vis"] == 0:
                    if best["hidden"] > 0:
                        activate(best)          # Rule 2: the next (drifted) slice, matched in this same submission
                        best["fresh"] = True    # untouched since activation (finding F6)
                    else:
                        book.remove(best)
            if inc["vis"] == 0 and inc["hidden"] > 0:
                activate(inc)
                continue
            break
        if inc["vis"] > 0 and ex in (NON, ICE):
            book.append(inc)                    # §6.2 item 4 / §4.5: what is left of a NON or ICE order rests
        # an IOC remainder or (impossible after the pre-check) a FOK remainder never rests
    return trades, book, fok_out, before_fok


def book_view(entries, skip=()):
    out = {"BUY": [], "SELL": []}
    for e in entries:
        if e["vis"] > 0 and e["oid"] not in skip:
            out[side_name(e["side"])].append([e["price"], e["vis"], e["oid"]])
    return {k: sorted(v) for k, v in out.items()}


def expect_run(tape):
    # Everything section 7 compares for one run, from (W) over the tape's rows.
    rows = real_rows(tape)
    oids = [int(r[C_OID]) for r in rows]
    s_f = fok_index(tape)
    fok_oid, req = int(rows[s_f][C_OID]), int(rows[s_f][C_QTY])
    trades, book, fok_out, _ = walk(tape)
    verdict, matchable = fok_out[fok_oid]
    filled = {o: 0 for o in oids}
    for t in trades:
        filled[t[0]] += t[3]
        filled[t[1]] += t[3]
    remaining = {o: 0 for o in oids}
    for e in book:
        remaining[e["oid"]] += e["vis"] + e["hidden"]
    fresh = [e["oid"] for e in book if e["fresh"]]
    books = [book_view(book)]
    if fresh:                                   # finding F6: a just-activated, untouched slice may not be shown yet
        books.append(book_view(book, skip=set(fresh)))
    exact = {
        "verdict": verdict,
        "accepted": sorted(oids), "rejected": [],
        "trades": trades,
        "filled": {str(o): filled[o] for o in oids},
        "remaining": {str(o): remaining[o] for o in oids},
        "cancelled": {str(fok_oid): [["fok_kill", req]]} if verdict == "killed" else {},
        "evicted": {},
    }
    if verdict == "killed":
        exact["fok_book_unchanged"] = True      # (R): finding F5
    return {"exact": exact, "allowed": {"final_book": books}, "req": req, "fok_oid": fok_oid, "s_fok": s_f,
            "matchable": matchable, "ice_oids": [int(r[C_OID]) for r in rows if int(r[C_EXEC]) == ICE],
            "fresh": fresh}


def ice_crossing_volume(ice, limit):
    # T_c = min(total, peak * r): the iceberg's crossing slices only (C10d).
    return min(int(ice[C_QTY]), int(ice[C_PEAK]) * crossing_slices(ice, limit))


def matchable_of(tape):
    # The pre-check volume of the tape's FOK against the book before it (section 5, "The pre-check").
    _, _, fok_out, _ = walk(tape)
    return int(next(iter(fok_out.values()))[1])


def run_requests(part, draw):
    # The run set of a draw (finding F3), from the tape: C10a {V-1, V, V+1}; C10b {req, T, T+1}; C10c {M, M+1};
    # C10d {T_c, T_c+1, M, M+1} (plus the drawn request, always one of them).
    rows = real_rows(draw.tape)
    m = matchable_of(draw.tape)
    req = int(rows[fok_index(draw.tape)][C_QTY])
    if part == "C10a":
        reqs = {m - 1, m, m + 1}
    elif part == "C10b":
        reqs = {req, m, m + 1}
    elif part == "C10c":
        reqs = {m, m + 1}
    else:
        tc = ice_crossing_volume(rows[0], rows[-1][C_PRICE])
        reqs = {req, tc, tc + 1, m, m + 1}
    return sorted(r for r in reqs if r >= 1)


def expectation(part, draw):
    # Every run of a draw: request -> expect_run(twin tape). Also the class.
    runs = {r: expect_run(twin(draw, r).tape) for r in run_requests(part, draw)}
    return {"runs": runs, "class": class_of(part, draw)}


# ---- the closed forms, from draw.params only ---------------------------------------------------------------------
def closed_form(part, p, req):
    # verdict, trades, filled and remaining of one run at request `req`, from draw.params (section 5).
    rest_side, fok_oid = int(p["rest_side"]), int(p["fok_oid"])
    s_f = fok_oid - 1                                   # the FOK is the last submission; oids are 1..N (section 4)

    def bs(maker):
        return (maker, fok_oid) if rest_side == BUY else (fok_oid, maker)

    if part == "C10a":
        qs, ps = [int(x) for x in p["level_qty_ints"]], [int(x) for x in p["level_price_ints"]]
        M = sum(qs)
        pieces = [(i + 1, ps[i], qs[i]) for i in range(len(qs))]       # level i: oid i + 1, at its own price
        total_rest = {str(i + 1): qs[i] for i in range(len(qs))}
    elif part == "C10d":
        T, pk, o_i, o_v = int(p["total_int"]), int(p["peak_int"]), int(p["ice_oid"]), int(p["vis_oid"])
        base, ppd, lim = int(p["base_int"]), int(p["ppd_int"]), int(p["p_fok_int"])
        pv, qv = int(p["vis_price_int"]), int(p["vis_qty_int"])
        n = ceil_div(T, pk)
        r = 0
        while r < n and crosses(rest_side, base + r * ppd, lim):
            r += 1
        # each piece with its rank (price, then entry): slice 0 entered before the visible order, every slice
        # k >= 1 is activated during the FOK, after it (finding F15).
        ranked = [(rank_key(rest_side, base + k * ppd, 0 if k == 0 else 2 + k), o_i, base + k * ppd,
                   min(pk, T - k * pk)) for k in range(r)]
        vis_crosses = crosses(rest_side, pv, lim)
        if vis_crosses:
            ranked.append((rank_key(rest_side, pv, 1), o_v, pv, qv))
        M = min(T, pk * r) + (qv if vis_crosses else 0)
        pieces = [(maker, price, x) for _, maker, price, x in sorted(ranked, key=lambda t: t[0])]
        total_rest = {str(o_i): T, str(o_v): qv}
    else:
        T, pk, o_i = int(p["total_int"]), int(p["peak_int"]), int(p["ice_oid"])
        base = int(p["p_ice_int"]) if part == "C10b" else int(p["base_int"])
        ppd = 0 if part == "C10b" else int(p["ppd_int"])
        n = ceil_div(T, pk)
        prices = [base + k * ppd for k in range(n)]
        lim = int(p["p_fok_int"])
        r = 0
        while r < n and crosses(rest_side, prices[r], lim):
            r += 1
        M = min(T, pk * r)                              # the C10c formula (C10b: every slice crosses, M = T)
        pieces = [(o_i, prices[k], min(pk, T - k * pk)) for k in range(r)]
        total_rest = {str(o_i): T}
    out = {"matchable": M, "verdict": "executed" if req <= M else "killed", "trades": []}
    filled = {**{k: 0 for k in total_rest}, str(fok_oid): 0}
    if req <= M:
        left = req
        for maker, price, x in pieces:
            if left == 0:
                break
            f = min(x, left)
            out["trades"].append([*bs(maker), price, f, s_f])
            filled[str(maker)] += f
            filled[str(fok_oid)] += f
            left -= f
    out["filled"] = filled
    out["remaining"] = {**{k: v - filled[k] for k, v in total_rest.items()}, str(fok_oid): 0}
    return out


def closed_form_agrees(part, draw):
    try:
        e = expectation(part, draw)
        for req, run in e["runs"].items():
            cf, x = closed_form(part, draw.params, req), run["exact"]
            if not (cf["verdict"] == x["verdict"] and cf["trades"] == x["trades"] and cf["matchable"] == run["matchable"]
                    and cf["filled"] == x["filled"] and cf["remaining"] == x["remaining"]):
                return False
        return True
    except Exception:  # noqa: BLE001  (a malformed params dict is a disagreement, recorded by the caller)
        return False


# ---- discrimination (finding F9): named wrong pre-checks, from the tape alone ------------------------------------
def wrong_readings(part, draw):
    # name -> the matchable volume a wrong pre-check would count on this draw's book.
    rows = real_rows(draw.tape)
    m = matchable_of(draw.tape)
    if part == "C10a":
        return {"one tick short (V - 1)": m - 1, "one tick long (V + 1)": m + 1}
    ice = rows[0]
    T, pk = int(ice[C_QTY]), int(ice[C_PEAK])
    if part == "C10d":
        qv = int(rows[1][C_QTY])
        out = {"iceberg only (the worse visible level ignored)": ice_crossing_volume(ice, rows[-1][C_PRICE]),
               "public book only (the iceberg's hidden quantity ignored)": min(pk, T) + qv,
               "one tick long (M + 1)": m + 1}
        if int(ice[C_PPD]) != 0:
            out["OQ-11 Interpretation A (all remaining iceberg quantity if the base crosses)"] = T + qv
        return out
    if part == "C10b":
        return {"visible peak only": min(pk, T), "one tick long (T + 1)": T + 1}
    r = crossing_slices(ice, rows[-1][C_PRICE])
    return {"OQ-11 Interpretation A (all remaining quantity if the base crosses)": T,
            "one slice too many": min(T, pk * (r + 1)), "one slice too few": min(T, pk * (r - 1))}


def undiscriminated(part, draw):
    # The wrong readings that give the rule's verdict on EVERY run of the draw (expected: none).
    reqs = run_requests(part, draw)
    m = matchable_of(draw.tape)
    right = [r <= m for r in reqs]
    return sorted(name for name, w in wrong_readings(part, draw).items() if [r <= w for r in reqs] == right)


# ---- the dataset precondition (section 4) ------------------------------------------------------------------------
def precondition(part, draw):
    # Returns the list of departures (empty: the draw is as section 4 describes).
    bad, t, p = [], np.asarray(draw.tape), draw.params
    rows = real_rows(t)
    if len(rows) < 2:
        return [f"{len(rows)} real rows"]
    rest, fok = rows[:-1], rows[-1]
    if len({int(x[C_CONTRACT]) for x in rows}) != 1:
        bad.append("more than one contract")
    oids, ts = [int(x[C_OID]) for x in rows], [int(x[C_TS]) for x in rows]
    if oids != list(range(1, len(rows) + 1)) or any(b <= a for a, b in zip(ts, ts[1:])):
        bad.append(f"order ids {oids} / tape timestamps {ts} not 1..N / strictly increasing")
    if any(not 0 <= int(x[C_PRICE]) <= PRICE_INT_MAX or not QTY_INT_MIN <= int(x[C_QTY]) <= QTY_INT_MAX for x in rows):
        bad.append("price or quantity outside §2.3")
    if any(int(x[C_VALIDITY]) != GFS for x in rows):
        bad.append("a row is not GFS")
    if int(fok[C_EXEC]) != FOK or int(fok[C_PEAK]) != 0 or int(fok[C_PPD]) != 0 or fok_index(t) != len(rows) - 1:
        bad.append("the last row is not a FOK without peak and increment")
    sides = {int(x[C_SIDE]) for x in rest}
    if len(sides) != 1 or int(fok[C_SIDE]) != other(next(iter(sides))):
        bad.append("the resting rows are not all on one side, opposite the FOK")
    rs = int(rest[0][C_SIDE])
    lim, req = int(fok[C_PRICE]), int(fok[C_QTY])
    common = {"rest_side": rs, "fok_oid": int(fok[C_OID]), "req_int": req, "p_fok_int": lim}
    if part == "C10d":
        if len(rest) != 2 or int(rest[0][C_EXEC]) != ICE or int(rest[1][C_EXEC]) != NON \
                or int(rest[1][C_PEAK]) != 0 or int(rest[1][C_PPD]) != 0:
            return bad + ["expected one ICE row then one plain NON row"]
        ice, vis = rest
        T, pk, base, ppd = int(ice[C_QTY]), int(ice[C_PEAK]), int(ice[C_PRICE]), int(ice[C_PPD])
        pv, qv = int(vis[C_PRICE]), int(vis[C_QTY])
        n = ceil_div(T, pk) if pk > 0 else 0
        if not 0 < pk < T or not 2 <= n <= SLICE_MARGIN:
            bad.append(f"iceberg peak {pk}, total {T}, ceil(total/peak) {n}")
        r = crossing_slices(ice, lim) if pk > 0 else 0
        if ppd == 0:
            if r != n:
                bad.append(f"standard iceberg: {r} of {n} slices cross the FOK limit (expected all)")
        else:
            if not sign_valid(rs, ppd):
                bad.append(f"increment {ppd} on a {side_name(rs)} iceberg is not sign-valid (§13.6)")
            cross = [crosses(rs, base + k * ppd, lim) for k in range(n)]
            if any(cross[r:]) or not 2 <= r <= n - 1:
                bad.append(f"crossing slices {cross} are not a prefix of length 2..n-1")
        if any(not 0 <= base + k * ppd <= PRICE_INT_MAX for k in range(n)):
            bad.append("a declared slice price base + k*ppd lies outside §2.3")
        if pv == base or not crosses(rs, base, pv) or not crosses(rs, pv, lim):
            bad.append(f"visible level {pv} is not strictly behind the base {base} and crossing the limit {lim}")
        tc = min(T, pk * r)
        if req not in (tc, tc + 1, tc + qv, tc + qv + 1):
            bad.append(f"request {req} not in T_c, T_c + 1, M, M + 1 (T_c = {tc}, M = {tc + qv})")
        vals = {**common, "ladder": "standard" if ppd == 0 else "icewpi", "ice_oid": int(ice[C_OID]),
                "base_int": base, "ppd_int": ppd, "peak_int": pk, "total_int": T, "n_slices": n,
                "reachable_slices": r, "vis_oid": int(vis[C_OID]), "vis_price_int": pv, "vis_qty_int": qv}
    elif part == "C10a":
        if any(int(x[C_EXEC]) != NON or int(x[C_PEAK]) != 0 or int(x[C_PPD]) != 0 for x in rest):
            bad.append("a resting row is not a plain NON")
        ps = [int(x[C_PRICE]) for x in rest]
        if any(not crosses(rs, a, b) or a == b for a, b in zip(ps, ps[1:])):
            bad.append(f"level prices {ps} are not strictly worse in submission order")
        if lim != ps[-1]:
            bad.append("the FOK limit is not the deepest level's price")
        V = sum(int(x[C_QTY]) for x in rest)
        if req not in (V - 1, V, V + 1):
            bad.append(f"request {req} not in V - 1..V + 1 (V = {V})")
        vals = {**common, "n_levels": len(rest), "level_price_ints": ps, "level_qty_ints": [int(x[C_QTY]) for x in rest],
                "v_int": V, "delta_ticks": req - V}
    else:
        if len(rest) != 1 or int(rest[0][C_EXEC]) != ICE:
            return bad + ["expected exactly one resting ICE row"]
        ice = rest[0]
        T, pk, base, ppd = int(ice[C_QTY]), int(ice[C_PEAK]), int(ice[C_PRICE]), int(ice[C_PPD])
        n = ceil_div(T, pk) if pk > 0 else 0
        if not 0 < pk < T or not 2 <= n <= SLICE_MARGIN:
            bad.append(f"iceberg peak {pk}, total {T}, ceil(total/peak) {n}")
        if part == "C10b":
            if ppd != 0:
                bad.append(f"C10b iceberg has increment {ppd} (expected 0)")
            if not crosses(rs, base, lim) or lim == base:
                bad.append("the FOK is not strictly through the iceberg's price")
            if not req > pk:
                bad.append(f"request {req} does not exceed the visible peak {pk}")
            vals = {**common, "ice_oid": int(ice[C_OID]), "p_ice_int": base, "total_int": T, "peak_int": pk, "n_slices": n}
        else:
            if ppd == 0 or not sign_valid(rs, ppd):
                bad.append(f"increment {ppd} on a {side_name(rs)} iceberg is not sign-valid (§13.6)")
            if any(not 0 <= base + k * ppd <= PRICE_INT_MAX for k in range(n)):
                bad.append("a declared slice price base + k*ppd lies outside §2.3")
            r = crossing_slices(ice, lim)
            cross = [crosses(rs, base + k * ppd, lim) for k in range(n)]
            if any(cross[r:]) or not 1 <= r <= n - 1:
                bad.append(f"crossing slices {cross} are not a non-empty proper prefix")
            elif lim != base + (r - 1) * ppd:
                bad.append("the FOK limit is not slice r - 1's price")
            vals = {**common, "ice_oid": int(ice[C_OID]), "base_int": base, "ppd_int": ppd, "peak_int": pk,
                    "total_int": T, "n_slices": n, "reachable_slices": r}
    off = sorted(k for k, v in vals.items() if p.get(k, "missing") != v)
    if off:
        bad.append(f"params != tape on {off}")
    if max_slices(t) > SLICE_MARGIN:
        bad.append(f"an iceberg row with ceil(qty/peak) > {SLICE_MARGIN}")
    return bad


# ---- the expectation on the first draw of each part ---------------------------------------------------------------
CF_AGREES = {}
for _p in PARTS:
    CF_AGREES[_p] = False
    if FIRST[_p] is None:
        continue
    _d = DRAWN[(_p, FIRST[_p])]
    _e = expectation(_p, _d)
    CF_AGREES[_p] = closed_form_agrees(_p, _d)
    print(f"{_p}, seed {FIRST[_p]} [{_e['class']}]: runs at requests {list(_e['runs'])} "
          f"(drawn request {int(real_rows(_d.tape)[-1][C_QTY])})")
    for _r, _x in _e["runs"].items():
        _k = _x["exact"]
        print(f"    req {_r} ({decode_qty(_r, CONFIG):.1f} MW): pre-check volume {_x['matchable']} -> {_k['verdict'].upper()}")
        print(f"        (Q) trades [buyer, seller, price_int, qty_int, submission]: {_k['trades']}")
        if _k["trades"]:
            print(f"            prices in EUR/MWh: {[round(decode_price(t[2], CONFIG), 2) for t in _k['trades']]}")
        print(f"        (F) filled {_k['filled']}; remaining {_k['remaining']}; (C) cancelled {_k['cancelled']}")
        print(f"        (B) final book one of: {_x['allowed']['final_book']}")
    print(f"    closed form == walk: {CF_AGREES[_p]}; precondition departures: {precondition(_p, _d)}; "
          f"undiscriminated wrong readings: {undiscriminated(_p, _d)}")
print(f"closed form agrees with (W) on the first draw of each part: {CF_AGREES}")

## 6. Run

### Eager and jitted (stub — to be wired at K3)

> **K2 stub.** This section is deliberately empty. The developer wires it at K3 and edits **nothing else** (independence rule 2): sections 1–5 and 7 are the expectation.

Section 7 consumes exactly **one name** from this section. While it is not defined, section 7 records a failing `section 6 wired` check, runs no draw, and the notebook ends `NOT CERTIFIED`.

**`run_draw(draw, *, jit) -> RunRecord`**: the battery's record builder on one draw, i.e. `tests.omiecid.reference.omie_invariants.record_from_draw(draw, jit=jit)`, with no other argument. It runs `draw.tape` **as is** (nothing in it may be rewritten) on one fresh book of an OPEN contract with the default configs (`OMIEProductConfig()`, `OMIEMarketConfig()`), the draw's own analytic capacity bounds (`draw.capacity_bounds`), no cancellations and no gate-closure expiry. `draw` is either a generator draw or a **twin** built by section 4's `twin(draw, req)` (a `Draw` whose tape differs only in the FOK row's quantity, with its own recomputed capacity bounds); both are run the same way. `jit=False` runs eager (the builder wraps the run in `jax.disable_jit()`); `jit=True` runs jitted. Every call starts from its own fresh book. It may raise; section 7 records a raise as a failed check.

Section 7 reads these `RunRecord` fields:
- `fills`: each `Fill`'s `submission`, `row_index`, `buyer_oid`, `seller_oid`, `price_int`, `qty_int`;
- `accepted` (its keys) and `rejections` (each `Rejection.oid`);
- `filled`, `still_resting`, `hidden_remaining`, `evicted` and `cancelled_or_expired` (each `Cancellation.cause` and `Cancellation.qty_int`), per order id;
- `pre_snapshots[s]` and `post_snapshots[s]` for the FOK's submission `s`: `bids`, `asks`, `trades`, `hidden_qty(oid)`, `book_full`, `truncated`, `ts_counter` (reported only);
- `final_snapshot.resting(side)`: `(price, qty, oid, agent, sec, nsec)`;
- `observed_capacity`;
- the whole record, through `run_battery` (each check's `status`, and I8's `evidence`), `check_m4_fok_dichotomy` and `compare_records`.

Section 7 calls `run_draw(..., jit=True)` on every run of every draw it uses, and `run_draw(..., jit=False)` on every run of the eager selection of section 3. A seed beyond the initial range (a re-draw after a discard) is prepared with `prepare(part, seed)` before the call.

In [ ]:
# ---------------------------------------------------------------------------------------
# Section 6 (K3): run_draw(draw, *, jit) -> RunRecord, per the contract above.
#
# The battery's record builder on one draw, with no other argument:
# record_from_draw runs draw.tape as is on one fresh book of one OPEN contract, with the
# default configs (OMIEProductConfig(), OMIEMarketConfig()), the draw's own analytic
# capacity bounds (draw.capacity_bounds; a twin carries its own, recomputed by section 4),
# no cancellations (no C10 draw schedules one) and no gate-closure expiry.
# jit=False wraps the whole run in jax.disable_jit() (inside build_record); jit=True runs
# it jitted. Every call starts from its own fresh book. It may raise; section 7 records
# the raise.
# ---------------------------------------------------------------------------------------
from tests.omiecid.reference.omie_invariants import record_from_draw


def run_draw(draw, *, jit):
    """One run of `draw.tape` (unmodified) on a fresh OPEN book: the battery's record builder."""
    return record_from_draw(draw, jit=jit)


print("run_draw: record_from_draw(draw, jit=jit) on one OPEN contract, default configs, the draw's capacity "
      "bounds, no cancels, no GCT expiry; jitted on every run of every draw, eager on the section-3 selection")


## 7. Certification

The oracle comparison comes first, by order id and in integer ticks, one combined assertion per valid draw; then the invariant battery (RB, I1–I9) as a pass/fail table, the capacity peaks and the discard rate. No check stops the notebook: every assertion is evaluated and recorded.

**What is recorded, per valid draw.** One assertion named by part, seed and class, whose `expected` and `observed` are the canonical compact JSON (sorted keys) of section 5's values for **every run of the draw, keyed by its request, in full**, never truncated, so an auditor can check every integer from the results JSON alone: the verdict; accepted and rejected ids; every trade row `[buyer, seller, price_int, qty_int, submission]` in row order; filled and remaining per id; cancellations `[cause, qty_int]` per id; evictions; the final book; on a kill, the rollback equality (finding F5). The final book is one of a listed set (finding F6): `expected` holds the whole set under `final_book_one_of`, and the observed book must be one of its members. The assertion's `detail` lists every `request:key` that differs, then, **reported and not asserted**, per run: whether the next slice was shown, and whether the engine's timestamp counter moved on a kill. The draw's tape and parameters are kept for a frozen counterexample (`record_case`).

**Aggregate assertions:** the section-6 wiring; the closed forms against (W), on the first draw of each part and on every valid draw; the dataset precondition (section 4), listing every departing seed and its departures; the **discrimination** of every named wrong pre-check on every valid draw (findings F9, F16); **M4**, **I3** and **I5** on every run, each failing **only** on a `status == "fail"` (every failing `part seed:request` listed in full), with `not_applicable` counted as vacuously satisfied — a killed FOK writes no trade, so I5 is `not_applicable` on it (§6.2 items 3 and 7 are vacuous) — and its count reported per part and expected verdict, every such run listed in full (finding F13); **no capacity flag inside the margin** (finding F10); the **hop bound** ≤ 160 and the largest $\lceil \mathit{qty}/\mathit{peak} \rceil$ ≤ 64 on every run, with the largest values reported; **coverage** — all eighteen classes among the valid draws, and both verdicts (executed and killed) observed by the expectation in every part; the eager comparisons, at least one per part and per class, a missing one recorded as a failure.

The battery runs on every jitted run with `expects_trade` true exactly when (V) says executed (a killed FOK trades nothing, and no resting order crosses another). **Reported, not asserted:** the `not_applicable` counts of I3, I5 and M4 (finding F13), the valid draws per class, the discards per reason, the eager seeds, how often the next slice was shown, and how often a kill moved the timestamp counter.

In [ ]:
import traceback
from collections import defaultdict

from tests.omiecid.reference.omie_invariants import (
    BatteryReport, CheckResult, check_m4_fok_dichotomy, compare_records, run_battery,
)

cert = Certification(
    cert_id=CERT_ID, notebook=NOTEBOOK, scenario="Fill-or-kill, in three parts",
    known_result="FOK pre-check volume M from the book on the other side (visible orders crossing the limit; every "
                 "iceberg slice whose drifted price crosses it: §4.2.1 Rule 1, OQ-11 B); executed iff req <= M, filling "
                 "exactly req at each maker's own (slice) price, uncapped; else killed: no trade, the book before == the "
                 "book after, the whole req cancelled as fok_kill (M4). C10a: M = V, runs V-1, V, V+1. C10b: M = total "
                 "(hidden quantity counts), runs req, total, total+1. C10c: M = min(total, peak * #{k >= 0: base + k*ppd "
                 "crosses the limit}), runs M, M+1. C10d (iceberg + a worse visible level): M = min(total, peak * r) + q_v, "
                 "runs T_c, T_c+1, M, M+1; fills in rank order (better price first; a slice tying the visible price "
                 "after it), each piece at its maker's price",
    oracle_kind="exact-parametric, plus M4", quotes=QUOTES, citations=CITATIONS,
    constructions=REGISTERED,   # C10d is notebook-local: no registry digest (finding F14)
    requested_draws=DRAWS * len(PARTS), runtime_forecast_s=560.0, started_at=T0,
    downstream="C11 (GCT expiry: no FOK-visible volume may remain for a closed contract), C14 (the FOK pre-check past "
               "64 slices, recorded not certified), C15 (fuzz sessions with FOK orders against icebergs), C17; the "
               "relation M4; Track B, whose agents may submit FOK orders against books holding hidden iceberg volume",
)

R_V = "§4.5 FOK row; §5.1 (FOK unmatched -> CANCELLED); §4.2.1 Rule 1; OQ-11 (Interpretation B)"
R_Q = "§4.2.1 Rules 1-2; §6.1; §6.2 items 2, 3, 6, 7 (each maker sets its price; no price capping)"
R_ALL = R_V + "; " + R_Q
R_DRAW = "§4.5; §5.1; §4.2.1 Rules 1-2 (incl. Re-ranking); §6.1; §6.2 items 2, 3, 6, 7; OQ-11 B; M4"   # short: repeated on every draw
R_M4 = "M4 (FOK fills exactly its request, or zero fills and a byte-identical book); §4.5"
R_I3 = "§4.2.1 Rule 4 (a crossed order book cannot persist); I3"
R_I5 = "§6.2 items 3 and 7 (the maker / the iceberg is the price-setter); I5 (not_applicable without a trade: vacuous)"
R_CAP = "capacity-validity precondition (five ceilings)"
R_DATA = "dataset precondition (section 4); §2.3; §13.6 sign rule (C10c)"
R_EAGER = "eager == jitted (scenario-ladder: what eager and jitted mean here)"
R_DISC = "rule 1 (every check seen failing): the draw discriminates each named wrong pre-check; OQ-11"

_SEVERITY = {"invalid": 3, "fail": 2, "pass": 1, "not_applicable": 0}


def merge_reports(reports):
    # One battery row per check for the whole draw: the worst status over its runs (invalid > fail > pass > n/a),
    # violations and unobserved pooled, evidence kept per run (section 3).
    out = []
    for i, first in enumerate(reports[0].results):
        rs = [rep.results[i] for rep in reports]
        status = max((r.status for r in rs), key=lambda st: _SEVERITY.get(st, 2))
        out.append(CheckResult(check_id=first.check_id, status=status,
                               violations=tuple(v for r in rs for v in r.violations),
                               unobserved=tuple(sorted({u for r in rs for u in r.unobserved})),
                               evidence={"runs": [dict(r.evidence) for r in rs]}))
    return BatteryReport(results=tuple(out))


def guarded(label, seed, fn, *args, **kw):
    # Call a section-6 function; a raise is recorded as a failed check, never propagated.
    try:
        return fn(*args, **kw)
    except Exception as exc:  # noqa: BLE001
        cert.check(f"{label}: run raised", False, seed=seed, expected="a RunRecord", observed=type(exc).__name__,
                   detail=f"{type(exc).__name__}: {exc} | " + traceback.format_exc().splitlines()[-1], rule=R_ALL)
        return None


def book3(snap):
    return {side_name(sd): sorted([int(x[0]), int(x[1]), int(x[2])] for x in snap.resting(sd)) for sd in (BUY, SELL)}


def book_struct(snap, ice_oids):
    # Finding F5: bid and ask arrays and the trade ring byte for byte, every iceberg's hidden quantity, the flags.
    arrs = tuple((np.asarray(getattr(snap, k)).tobytes(), np.asarray(getattr(snap, k)).shape, str(np.asarray(getattr(snap, k)).dtype))
                 for k in ("bids", "asks", "trades"))
    return (arrs, tuple(int(snap.hidden_qty(o)) for o in ice_oids), bool(snap.book_full), bool(snap.truncated))


def observe(rec, x):
    # The observed counterpart of one run's expectation x (exact + allowed), plus reported-only info.
    fills = sorted(rec.fills, key=lambda f: (int(f.submission), int(f.row_index)))
    oids = [int(o) for o in x["exact"]["filled"]]
    fok, req, s_f = x["fok_oid"], x["req"], x["s_fok"]
    filled = {str(o): int(rec.filled.get(o, 0)) for o in oids}
    canc = {str(int(o)): [[str(c.cause), int(c.qty_int)] for c in cs] for o, cs in sorted(rec.cancelled_or_expired.items()) if cs}
    fok_rows = [f for f in fills if int(f.submission) == s_f]
    if filled[str(fok)] == req and str(fok) not in canc:
        verdict = "executed"
    elif filled[str(fok)] == 0 and not fok_rows:
        verdict = "killed"
    else:
        verdict = f"partial ({filled[str(fok)]} of {req})"
    o = {
        "verdict": verdict,
        "accepted": sorted(int(a) for a in rec.accepted),
        "rejected": sorted(int(r.oid) for r in rec.rejections),
        "trades": [[int(f.buyer_oid), int(f.seller_oid), int(f.price_int), int(f.qty_int), int(f.submission)] for f in fills],
        "filled": filled,
        "remaining": {str(q): int(rec.still_resting.get(q, 0)) + int(rec.hidden_remaining.get(q, 0)) for q in oids},
        "cancelled": canc,
        "evicted": {str(int(q)): int(v) for q, v in sorted(rec.evicted.items()) if int(v)},
        "final_book": book3(rec.final_snapshot),
    }
    info = {}
    if "fok_book_unchanged" in x["exact"]:
        pre, post = rec.pre_snapshots[s_f], rec.post_snapshots[s_f]
        o["fok_book_unchanged"] = book_struct(pre, x["ice_oids"]) == book_struct(post, x["ice_oids"])
        info["ts_counter_moved_on_kill"] = int(post.ts_counter) != int(pre.ts_counter)
    if x["fresh"]:
        info["next_slice"] = ("shown" if any(r[2] in x["fresh"] for sd in o["final_book"].values() for r in sd)
                              else "not shown")
    return o, info


EAGER_DONE = defaultdict(list)   # part or class -> seeds with an eager comparison


def margin_breaches(tape, bounds):
    # The capacity-validity precondition (five ceilings), checked from a run's tape before it is run.
    lim = {"resting_per_side": ENVELOPE.max_resting_per_side, "trades": ENVELOPE.max_trades,
           "hops": ENVELOPE.max_hops, "concurrent_icebergs": ENVELOPE.max_concurrent_icebergs}
    bad = []
    for name, bd in (("draw", bounds), ("generic", synth.tape_capacity_bounds(tape))):
        bad += [f"{name} {k} bound {bd.get(k)} > {m}" for k, m in lim.items() if int(bd.get(k, 0)) > m]
    if max_slices(tape) > SLICE_MARGIN:
        bad.append(f"ceil(qty/peak) {max_slices(tape)} > {SLICE_MARGIN}")
    return bad


for _p in PARTS:
    cert.check(f"{_p} section 5: closed form agrees with the walk (W) on the first draw", CF_AGREES[_p],
               expected=True, observed=CF_AGREES[_p], unit="bool", rule=R_ALL)
wired = callable(globals().get("run_draw"))
cert.check("section 6 wired (run_draw defined)", wired, expected=True, observed=wired, unit="bool",
           detail="" if wired else "section 6 is still the K2 stub", rule=R_ALL)

PER_DRAW, DATA_BAD, CLOSED_BAD, NO_DISCRIM = [], {}, [], {}
VALID = defaultdict(list)
COVER = Counter()
VERDICTS = defaultdict(Counter)            # part -> expected verdict counts over the runs of valid draws
FLAG_INSIDE_MARGIN = {}                    # "part seed:req" -> I8 evidence, for runs that passed the pre-run margin
BAT_FAIL = defaultdict(list)               # check id -> "part seed:req" whose status == "fail" (I3, I5, M4)
BAT_NA = defaultdict(list)                 # check id -> "part seed:req (verdict)" whose status == "not_applicable" (F13)
NA_COUNT = Counter()                       # (check id, part, expected verdict) -> not_applicable runs (F13)
DISCARDS = Counter()
PEAK = {"hop_bound": 0, "generic_hop_bound": 0, "max_slices": 0}
SHOWN = Counter()
TS_MOVED = Counter()


def check_draw(part, seed, e, recs):
    # (V), (A), (Q), (F), (C), (B), (R) of section 5 for every run of one draw. Returns ok.
    expected, observed, diff, info = {}, {}, [], {}
    for req, x in e["runs"].items():
        o, inf = observe(recs[req], x)
        diff += [f"{req}:{k}" for k, v in x["exact"].items() if o.get(k) != v]
        diff += [f"{req}:{k}" for k, allowed in x["allowed"].items() if o.get(k) not in allowed]
        expected[str(req)] = {**x["exact"], **{f"{k}_one_of": v for k, v in x["allowed"].items()}}
        observed[str(req)] = o
        if inf:
            info[str(req)] = inf
            if "next_slice" in inf:
                SHOWN[inf["next_slice"]] += 1
            if "ts_counter_moved_on_kill" in inf:
                TS_MOVED[inf["ts_counter_moved_on_kill"]] += 1
    return cert.check(
        f"seed {seed} [{e['class']}], requests {list(e['runs'])}", not diff, seed=seed,
        expected=canon(expected), observed=canon(observed), unit="other",
        detail=("differs on: " + ", ".join(diff) + (" | " if info else "") if diff else "")
               + ("reported only: " + canon(info) if info else ""), rule=R_DRAW)


def eager_compare(part, seed, runs, recs, tags):
    # Run every run of the draw eager and compare each with its jitted record (one eager_jit assertion).
    diffs = {}
    for req, d in runs.items():
        rec_e = guarded(f"{part} seed {seed} req {req} (eager)", seed, run_draw, d, jit=False)
        rec = recs.get(req)
        diffs[str(req)] = (["jitted run missing" if rec is None else "eager run missing"] if rec is None or rec_e is None
                           else list(compare_records(rec_e, rec)))
    for t in tags:
        EAGER_DONE[t].append(seed)
    ok = all(not v for v in diffs.values())
    cert.check(f"eager == jitted ({part} seed {seed}, every run)", ok, kind="eager_jit", seed=seed,
               expected=True, observed=ok, unit="bool",
               detail="identical" if ok else canon({k: v for k, v in diffs.items() if v}), rule=R_EAGER)


def run_part(part):
    valid, seed, eager_n = 0, SEED0[part], 0
    while valid < DRAWS and seed < SEED0[part] + 2 * DRAWS:
        draw = DRAWN.get((part, seed), "missing")
        if draw == "missing":
            draw = prepare(part, seed)
        if draw is None:
            cert.discard(seed, f"{part}: generator refused the seed: {REFUSED.get((part, seed))}")
            DISCARDS[f"{part} generator refused"] += 1
            seed += 1
            continue
        bad = precondition(part, draw)
        if bad:
            DATA_BAD[f"{part} seed {seed}"] = bad
            cert.check(f"{part} seed {seed}: draw is as section 4 describes", False, seed=seed,
                       expected=[], observed=bad, unit="other", rule=R_DATA)
            seed += 1
            continue
        e = expectation(part, draw)
        runs = {req: twin(draw, req) for req in e["runs"]}
        margin = {req: margin_breaches(d.tape, dict(d.capacity_bounds)) for req, d in runs.items()}
        margin = {r: m for r, m in margin.items() if m}
        if margin:
            cert.discard(seed, f"{part}: pre-run capacity margin: " + canon({str(r): m for r, m in margin.items()}))
            DISCARDS[f"{part} pre-run margin"] += 1
            seed += 1
            continue
        recs = {req: guarded(f"{part} seed {seed} req {req} (jit)", seed, run_draw, d, jit=True) for req, d in runs.items()}
        if any(r is None for r in recs.values()):
            seed += 1
            continue
        reps = {req: run_battery(rec, expects_trade=e["runs"][req]["exact"]["verdict"] == "executed")
                for req, rec in recs.items()}
        i8_bad = {req: dict(next(r for r in rep.results if r.check_id == "I8").evidence)
                  for req, rep in reps.items() if next(r for r in rep.results if r.check_id == "I8").status == "invalid"}
        if not cert.add_battery(merge_reports(list(reps.values())), seed=seed):
            for req, ev in i8_bad.items():
                FLAG_INSIDE_MARGIN[f"{part} {seed}:{req}"] = ev
            DISCARDS[f"{part} I8 invalid"] += 1
            seed += 1   # discarded and re-drawn with the next seed (and a failed assertion: finding F10)
            continue
        valid += 1
        VALID[part].append(seed)
        cert.record_case(seed, draw.tape, {**draw.params, "construction": CONSTRUCTION[part],
                                           "run_requests": list(e["runs"])})
        for req, d in runs.items():
            PEAK["hop_bound"] = max(PEAK["hop_bound"], int(d.hop_bound))
            PEAK["generic_hop_bound"] = max(PEAK["generic_hop_bound"], int(synth.tape_capacity_bounds(d.tape)["hops"]))
            PEAK["max_slices"] = max(PEAK["max_slices"], max_slices(d.tape))
            _verdict = e["runs"][req]["exact"]["verdict"]
            VERDICTS[part][_verdict] += 1
            m4 = check_m4_fok_dichotomy(recs[req], fok_oid=e["runs"][req]["fok_oid"])
            for r in [x for x in reps[req].results if x.check_id in ("I3", "I5")] + [m4]:
                # finding F13: only "fail" fails; "not_applicable" is vacuously satisfied, counted and reported
                if r.status == "fail":
                    BAT_FAIL[r.check_id].append(f"{part} {seed}:{req}")
                elif r.status == "not_applicable":
                    BAT_NA[r.check_id].append(f"{part} {seed}:{req} ({_verdict})")
                    NA_COUNT[(r.check_id, part, _verdict)] += 1
            cert.record_capacity({**recs[req].observed_capacity, "hops": None})   # hops: unobserved by the engine
        COVER[e["class"]] += 1
        if not closed_form_agrees(part, draw):
            CLOSED_BAD.append(f"{part} {seed}")
        und = undiscriminated(part, draw)
        if und:
            NO_DISCRIM[f"{part} {seed}"] = und
        ok = check_draw(part, seed, e, recs)
        PER_DRAW.append({"part": part, "seed": seed, "class": e["class"], "ok": ok})
        if valid <= EAGER_DRAWS or e["class"] not in EAGER_DONE:
            eager_compare(part, seed, runs, recs, [part, e["class"]])
        seed += 1
    return valid


N_VALID = {p: (run_part(p) if wired else 0) for p in PARTS}

# ---- aggregate assertions (every list in full) ----------------------------------------------------------------
_nv = sum(N_VALID.values())
cert.check("section 5: closed form agrees with the walk (W) on every valid draw", wired and not CLOSED_BAD,
           expected=[], observed=CLOSED_BAD, unit="other", rule=R_ALL)
cert.check("dataset: every draw is as section 4 describes (precondition)", wired and not DATA_BAD,
           expected={}, observed=DATA_BAD, unit="other",
           detail=f"{len(DATA_BAD)} draws depart" if DATA_BAD else f"{_nv} valid draws", rule=R_DATA)
cert.check("discrimination: every named wrong pre-check gives a different verdict on some run of every valid draw (F9)",
           wired and _nv > 0 and not NO_DISCRIM, expected={}, observed=NO_DISCRIM, unit="other",
           detail=f"valid draws: {_nv}", rule=R_DISC)


def na_summary(cid):
    # finding F13: the not_applicable count of one check, per part and expected verdict (reported, not asserted).
    by = {f"{p} {v}": NA_COUNT[(cid, p, v)] for p in PARTS for v in ("executed", "killed") if NA_COUNT[(cid, p, v)]}
    return f"not_applicable (vacuously satisfied) on {len(BAT_NA[cid])} runs: {canon(by)}"


for _cid, _rule in (("M4", R_M4), ("I3", R_I3), ("I5", R_I5)):
    cert.check(f"{_cid}: no run of any valid draw returns status 'fail' (not_applicable counts as satisfied: F13)",
               wired and _nv > 0 and not BAT_FAIL[_cid], expected=[], observed=BAT_FAIL[_cid], unit="other",
               detail=f"valid draws: {_nv}; failing runs: {len(BAT_FAIL[_cid])}; {na_summary(_cid)}", rule=_rule)
cert.check("no capacity flag (truncated, book_full, saturated ring) on a run inside the analytic margin (F10)",
           wired and not FLAG_INSIDE_MARGIN, expected={}, observed=FLAG_INSIDE_MARGIN, unit="other", rule=R_CAP)
cert.check("hop bound <= 160 on every run (largest reported: draw bound, generic bound)",
           wired and _nv > 0 and PEAK["hop_bound"] <= ENVELOPE.max_hops and PEAK["generic_hop_bound"] <= ENVELOPE.max_hops,
           expected=f"<= {ENVELOPE.max_hops}", observed=[PEAK["hop_bound"], PEAK["generic_hop_bound"]],
           unit="count", rule=R_CAP)
cert.check("largest ceil(qty/peak) over every run's icebergs <= 64 (largest reported)",
           wired and 0 < PEAK["max_slices"] <= SLICE_MARGIN, expected=f"<= {SLICE_MARGIN}",
           observed=PEAK["max_slices"], unit="count", rule=R_CAP)
_got = {c: COVER[c] for c in ALL_CLASSES}
_missing = [c for c, v in _got.items() if v == 0]
cert.check("dataset: the valid draws cover every class (section 3)", wired and not _missing,
           expected={c: ">= 1" for c in ALL_CLASSES}, observed=_got, unit="count",
           detail=f"missing: {_missing}" if _missing else "", rule=R_DATA)
_both = {p: dict(VERDICTS[p]) for p in PARTS}
cert.check("dataset: both verdicts (executed and killed) are expected on some run of every part",
           wired and all(VERDICTS[p]["executed"] > 0 and VERDICTS[p]["killed"] > 0 for p in PARTS),
           expected={p: {"executed": ">= 1", "killed": ">= 1"} for p in PARTS}, observed=_both, unit="count", rule=R_V)

# ---- eager: every part and every class must have had one comparison (absence = failure) -----------------------
for _t in list(PARTS) + list(ALL_CLASSES):
    if not EAGER_DONE.get(_t):
        cert.check(f"eager == jitted for '{_t}'", False, kind="eager_jit", expected=True, observed=None, unit="bool",
                   detail=f"no eager comparison was run for '{_t}'", rule=R_EAGER)

# ---- reports (not asserted) ------------------------------------------------------------------------------------
_fail_runs = [r for r in PER_DRAW if not r["ok"]]
display(Markdown(f"**Per-draw summary.** {len(PER_DRAW)} draws checked, {len(_fail_runs)} with a failed assertion"
                 + (": " + ", ".join(f"{r['part']} seed {r['seed']}" for r in _fail_runs) if _fail_runs else ".")))
print(f"valid draws per part: {N_VALID}; per class: {dict(COVER)}")
print(f"expected verdicts over the runs, per part: {_both}")
for _cid in ("M4", "I3", "I5"):
    print(f"{_cid}: {na_summary(_cid)}; every such run: {BAT_NA[_cid]}")
print(f"discards per reason: {dict(DISCARDS)}")
print(f"eager comparisons run: { {k: v for k, v in EAGER_DONE.items()} }")
print(f"next slice after an execution ending on a slice boundary (finding F6; reported only): {dict(SHOWN)}")
print(f"engine timestamp counter moved on a kill (finding F5; reported only): {dict(TS_MOVED)}")
print(f"analytic peaks over valid runs: hop bound {PEAK['hop_bound']} (generic {PEAK['generic_hop_bound']}), "
      f"largest ceil(qty/peak) {PEAK['max_slices']}")
display(Markdown(cert.table()))

## 8. Verdict

Emit the results JSON (validated against `results.schema.json`) and print exactly one word:
`PASS` or `NOT CERTIFIED`. On `NOT CERTIFIED` the smallest failing case is frozen as a fixture.

In [ ]:
RESULTS_PATH = cert.emit()
print(cert.verdict_line())
print(f"results -> {RESULTS_PATH}")

In [ ]:
# Section 9 ("Why this certification failed") is rendered here only on NOT CERTIFIED; it is
# persisted as the last markdown cell at K6-F by cert_support.append_failure_section.
_section9 = cert.failure_section_markdown()
if _section9:
    display(Markdown(_section9))